# Java 11 to 21+ Migration Agent System

An agentic system that automates the migration of Maven-based Java projects from Java 11 to 21+, including Spring Boot upgrades and dependency migrations.

In [53]:
# Install required dependencies
# !pip install langchain>=0.1.0 langchain-openai>=0.0.5 langgraph>=0.0.40 langchain-community>=0.0.20 openai>=1.0.0 redis>=5.0.0 gitpython>=3.1.0 pydantic>=2.0.0 requests>=2.31.0 filelock>=3.12.0

In [54]:
# Enhanced imports with LangChain tools and LangGraph
import os
import json
import subprocess
import re
import time
import threading
from typing import Dict, List, Optional, Any, Union, TypedDict, Annotated
from datetime import datetime
from enum import Enum
from dataclasses import dataclass, asdict
from abc import ABC, abstractmethod
from pathlib import Path

# Core dependencies
from pydantic import BaseModel, Field
import requests
from filelock import FileLock

# LangChain imports
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationSummaryBufferMemory
from langchain.memory.chat_message_histories import RedisChatMessageHistory
from langchain.schema import AgentAction, AgentFinish

# LangGraph imports
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import create_react_agent
# from langgraph.checkpoint.sqlite import SqliteSaver

import git
import redis
from dotenv import load_dotenv

load_dotenv()

# Configuration
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'your-api-key-here')
REDIS_URL = os.getenv('REDIS_URL', 'redis://localhost:6379')
BING_SEARCH_API_KEY = os.getenv('BING_SEARCH_API_KEY', 'your-bing-api-key')
MAVEN_CENTRAL_API_BASE = "https://search.maven.org/solrsearch/select"

# LLM Configuration
CLAUDE_SONNET = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=OPENAI_API_KEY)
CLAUDE_HAIKU = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=OPENAI_API_KEY)

print("✅ Enhanced imports loaded successfully!")

✅ Enhanced imports loaded successfully!


In [55]:
# Core Data Models and Enums

class RiskLevel(Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"

class MigrationPhase(Enum):
    ANALYSIS = "analysis"
    DEPENDENCY_RESOLUTION = "dependency_resolution"
    JAVA_VERSION_UPGRADE = "java_version_upgrade"
    SPRING_BOOT_UPGRADE = "spring_boot_upgrade"
    JAVAX_TO_JAKARTA = "javax_to_jakarta"
    JUNIT_MIGRATION = "junit_migration"
    TESTING_VALIDATION = "testing_validation"
    CUSTOM_PATTERNS = "custom_patterns"
    COMPLETED = "completed"

class Decision(BaseModel):
    timestamp: str
    context: str
    file_path: str
    issue: str
    options: List[Dict[str, Any]]
    chosen_option: str
    rationale: str
    risk_level: RiskLevel

class MigrationState(BaseModel):
    project_path: str
    current_phase: MigrationPhase
    completed_steps: List[str]
    failed_attempts: Dict[str, List[str]]
    checkpoints: List[str]  # git commit hashes
    human_decisions: List[Decision]
    risk_assessment: Dict[str, str]
    session_id: str
    started_at: str
    last_updated: str

# LangGraph State Definition
class AgentState(TypedDict):
    messages: Annotated[List[Dict[str, str]], add_messages]
    project_path: str
    session_id: str
    current_phase: str
    error_count: int
    last_action_result: str
    escalation_needed: bool

@dataclass
class EscalationContext:
    escalation_type: str
    context: Dict[str, Any]
    recommendation: str
    risk_level: RiskLevel
    file_path: Optional[str] = None
    current_code: Optional[str] = None
    options: Optional[List[Dict[str, Any]]] = None

print("✅ Data models defined successfully!")

✅ Data models defined successfully!


In [56]:
# Enhanced Dependency Management with Maven Central Integration

import xml.etree.ElementTree as ET
from typing import Dict, List, Optional, NamedTuple
import requests
import re
from packaging import version

class Dependency(NamedTuple):
    """Represents a Maven dependency."""
    group_id: str
    artifact_id: str
    current_version: str
    scope: Optional[str] = None

class Plugin(NamedTuple):
    """Represents a Maven plugin."""
    group_id: str
    artifact_id: str
    current_version: str

class MavenCentralClient:
    """Client for interacting with Maven Central API."""
    
    def __init__(self):
        self.base_url = "https://search.maven.org/solrsearch/select"
        self.session = requests.Session()
        self.session.timeout = 10
    
    def get_latest_version(self, group_id: str, artifact_id: str) -> Optional[str]:
        """Get the latest version of an artifact from Maven Central."""
        try:
            params = {
                'q': f'g:"{group_id}" AND a:"{artifact_id}"',
                'core': 'gav',
                'rows': 1,
                'wt': 'json',
                'sort': 'timestamp desc'
            }
            
            response = self.session.get(self.base_url, params=params)
            response.raise_for_status()
            
            data = response.json()
            docs = data.get('response', {}).get('docs', [])
            
            if docs:
                return docs[0].get('v')
            
            return None
            
        except Exception as e:
            print(f"Error fetching version for {group_id}:{artifact_id}: {e}")
            return None
    
    def get_all_versions(self, group_id: str, artifact_id: str, limit: int = 20) -> List[str]:
        """Get all available versions of an artifact."""
        try:
            params = {
                'q': f'g:"{group_id}" AND a:"{artifact_id}"',
                'core': 'gav',
                'rows': limit,
                'wt': 'json'
            }
            
            response = self.session.get(self.base_url, params=params)
            response.raise_for_status()
            
            data = response.json()
            docs = data.get('response', {}).get('docs', [])
            
            versions = [doc.get('v') for doc in docs if doc.get('v')]
            return sorted(versions, key=lambda v: version.parse(v) if self._is_valid_version(v) else version.parse("0.0.0"), reverse=True)
            
        except Exception as e:
            print(f"Error fetching versions for {group_id}:{artifact_id}: {e}")
            return []
    
    def _is_valid_version(self, version_str: str) -> bool:
        """Check if a version string is valid for parsing."""
        try:
            version.parse(version_str)
            return True
        except:
            return False

class PomParser:
    """Parser for Maven pom.xml files."""
    
    def __init__(self, pom_path: str):
        self.pom_path = pom_path
        self.tree = None
        self.root = None
        self.namespaces = {'maven': 'http://maven.apache.org/POM/4.0.0'}
        self._load_pom()
    
    def _load_pom(self):
        """Load and parse the pom.xml file."""
        try:
            self.tree = ET.parse(self.pom_path)
            self.root = self.tree.getroot()
            
            # Handle default namespace
            if self.root.tag.startswith('{'):
                namespace = self.root.tag.split('}')[0] + '}'
                self.namespaces['maven'] = namespace[1:-1]
        except Exception as e:
            print(f"Error loading pom.xml: {e}")
            raise
    
    def get_dependencies(self) -> List[Dependency]:
        """Extract all dependencies from pom.xml."""
        dependencies = []
        
        # Find all dependency elements
        for dep_elem in self.root.findall('.//maven:dependency', self.namespaces):
            group_id = self._get_element_text(dep_elem, 'maven:groupId')
            artifact_id = self._get_element_text(dep_elem, 'maven:artifactId')
            version = self._get_element_text(dep_elem, 'maven:version')
            scope = self._get_element_text(dep_elem, 'maven:scope')
            
            if group_id and artifact_id and version:
                # Resolve version variables
                version = self._resolve_property(version)
                dependencies.append(Dependency(group_id, artifact_id, version, scope))
        
        return dependencies
    
    def get_plugins(self) -> List[Plugin]:
        """Extract all plugins from pom.xml."""
        plugins = []
        
        # Find all plugin elements
        for plugin_elem in self.root.findall('.//maven:plugin', self.namespaces):
            group_id = self._get_element_text(plugin_elem, 'maven:groupId')
            artifact_id = self._get_element_text(plugin_elem, 'maven:artifactId')
            version = self._get_element_text(plugin_elem, 'maven:version')
            
            # Default group_id for Maven plugins
            if not group_id:
                group_id = 'org.apache.maven.plugins'
            
            if artifact_id and version:
                # Resolve version variables
                version = self._resolve_property(version)
                plugins.append(Plugin(group_id, artifact_id, version))
        
        return plugins
    
    def get_properties(self) -> Dict[str, str]:
        """Extract all properties from pom.xml."""
        properties = {}
        
        props_elem = self.root.find('.//maven:properties', self.namespaces)
        if props_elem is not None:
            for prop in props_elem:
                # Remove namespace prefix from tag name
                tag_name = prop.tag.split('}')[-1] if '}' in prop.tag else prop.tag
                properties[tag_name] = prop.text or ''
        
        return properties
    
    def update_dependency_version(self, group_id: str, artifact_id: str, new_version: str) -> bool:
        """Update a dependency version in the pom.xml."""
        try:
            for dep_elem in self.root.findall('.//maven:dependency', self.namespaces):
                dep_group = self._get_element_text(dep_elem, 'maven:groupId')
                dep_artifact = self._get_element_text(dep_elem, 'maven:artifactId')
                
                if dep_group == group_id and dep_artifact == artifact_id:
                    version_elem = dep_elem.find('maven:version', self.namespaces)
                    if version_elem is not None:
                        old_version = version_elem.text
                        version_elem.text = new_version
                        print(f"Updated {group_id}:{artifact_id} from {old_version} to {new_version}")
                        return True
            
            return False
            
        except Exception as e:
            print(f"Error updating dependency version: {e}")
            return False
    
    def update_plugin_version(self, group_id: str, artifact_id: str, new_version: str) -> bool:
        """Update a plugin version in the pom.xml."""
        try:
            for plugin_elem in self.root.findall('.//maven:plugin', self.namespaces):
                plugin_group = self._get_element_text(plugin_elem, 'maven:groupId') or 'org.apache.maven.plugins'
                plugin_artifact = self._get_element_text(plugin_elem, 'maven:artifactId')
                
                if plugin_group == group_id and plugin_artifact == artifact_id:
                    version_elem = plugin_elem.find('maven:version', self.namespaces)
                    if version_elem is not None:
                        old_version = version_elem.text
                        version_elem.text = new_version
                        print(f"Updated plugin {group_id}:{artifact_id} from {old_version} to {new_version}")
                        return True
            
            return False
            
        except Exception as e:
            print(f"Error updating plugin version: {e}")
            return False
    
    def save_pom(self) -> bool:
        """Save the modified pom.xml back to disk."""
        try:
            # Format the XML properly
            self._indent_xml(self.root)
            self.tree.write(self.pom_path, encoding='utf-8', xml_declaration=True)
            return True
        except Exception as e:
            print(f"Error saving pom.xml: {e}")
            return False
    
    def _get_element_text(self, parent, tag) -> Optional[str]:
        """Get text content of a child element."""
        elem = parent.find(tag, self.namespaces)
        return elem.text if elem is not None else None
    
    def _resolve_property(self, value: str) -> str:
        """Resolve property placeholders like ${property.name}."""
        if not value or not value.startswith('${'):
            return value
        
        # Extract property name
        match = re.match(r'\$\{([^}]+)\}', value)
        if not match:
            return value
        
        prop_name = match.group(1)
        properties = self.get_properties()
        
        return properties.get(prop_name, value)
    
    def _indent_xml(self, elem, level=0):
        """Add proper indentation to XML for readable output."""
        indent = "\n" + level * "  "
        if len(elem):
            if not elem.text or not elem.text.strip():
                elem.text = indent + "  "
            if not elem.tail or not elem.tail.strip():
                elem.tail = indent
            for child in elem:
                self._indent_xml(child, level + 1)
            if not child.tail or not child.tail.strip():
                child.tail = indent
        else:
            if level and (not elem.tail or not elem.tail.strip()):
                elem.tail = indent

@tool
def analyze_and_update_dependencies(project_path: str) -> str:
    """Analyze all dependencies and plugins, check Maven Central for updates, and apply them."""
    pom_path = os.path.join(project_path, "pom.xml")
    
    if not os.path.exists(pom_path):
        return "❌ pom.xml not found in project"
    
    try:
        # Parse the pom.xml
        parser = PomParser(pom_path)
        maven_client = MavenCentralClient()
        
        results = []
        updates_applied = 0
        
        # Check dependencies
        results.append("🔍 Analyzing Dependencies:")
        dependencies = parser.get_dependencies()
        
        for dep in dependencies:
            if dep.scope in ['test', 'provided']:
                continue  # Skip test and provided dependencies for now
            
            latest_version = maven_client.get_latest_version(dep.group_id, dep.artifact_id)
            
            if latest_version and latest_version != dep.current_version:
                try:
                    # Check if latest version is actually newer
                    if version.parse(latest_version) > version.parse(dep.current_version):
                        results.append(f"  📦 {dep.group_id}:{dep.artifact_id}")
                        results.append(f"      Current: {dep.current_version}")
                        results.append(f"      Latest:  {latest_version}")
                        
                        # Apply the update
                        if parser.update_dependency_version(dep.group_id, dep.artifact_id, latest_version):
                            results.append(f"      ✅ Updated successfully")
                            updates_applied += 1
                        else:
                            results.append(f"      ❌ Update failed")
                    else:
                        results.append(f"  ✅ {dep.group_id}:{dep.artifact_id} is up to date ({dep.current_version})")
                        
                except Exception as e:
                    results.append(f"  ⚠️ Version comparison failed for {dep.group_id}:{dep.artifact_id}: {e}")
            else:
                results.append(f"  ✅ {dep.group_id}:{dep.artifact_id} ({dep.current_version}) - No update available")
        
        # Check plugins
        results.append("\n🔧 Analyzing Plugins:")
        plugins = parser.get_plugins()
        
        for plugin in plugins:
            latest_version = maven_client.get_latest_version(plugin.group_id, plugin.artifact_id)
            
            if latest_version and latest_version != plugin.current_version:
                try:
                    if version.parse(latest_version) > version.parse(plugin.current_version):
                        results.append(f"  🔌 {plugin.group_id}:{plugin.artifact_id}")
                        results.append(f"      Current: {plugin.current_version}")
                        results.append(f"      Latest:  {latest_version}")
                        
                        # Apply the update
                        if parser.update_plugin_version(plugin.group_id, plugin.artifact_id, latest_version):
                            results.append(f"      ✅ Updated successfully")
                            updates_applied += 1
                        else:
                            results.append(f"      ❌ Update failed")
                    else:
                        results.append(f"  ✅ {plugin.group_id}:{plugin.artifact_id} is up to date ({plugin.current_version})")
                        
                except Exception as e:
                    results.append(f"  ⚠️ Version comparison failed for {plugin.group_id}:{plugin.artifact_id}: {e}")
            else:
                results.append(f"  ✅ {plugin.group_id}:{plugin.artifact_id} ({plugin.current_version}) - No update available")
        
        # Save the updated pom.xml if any updates were made
        if updates_applied > 0:
            if parser.save_pom():
                results.append(f"\n✅ Successfully applied {updates_applied} updates to pom.xml")
            else:
                results.append(f"\n❌ Failed to save updated pom.xml")
        else:
            results.append(f"\n✅ All dependencies and plugins are up to date")
        
        return "\n".join(results)
        
    except Exception as e:
        return f"❌ Error analyzing dependencies: {str(e)}"

@tool
def check_specific_dependency(project_path: str, group_id: str, artifact_id: str) -> str:
    """Check a specific dependency for updates and show all available versions."""
    pom_path = os.path.join(project_path, "pom.xml")
    
    if not os.path.exists(pom_path):
        return "❌ pom.xml not found in project"
    
    try:
        parser = PomParser(pom_path)
        maven_client = MavenCentralClient()
        
        # Find the dependency in pom.xml
        dependencies = parser.get_dependencies()
        target_dep = None
        
        for dep in dependencies:
            if dep.group_id == group_id and dep.artifact_id == artifact_id:
                target_dep = dep
                break
        
        if not target_dep:
            return f"❌ Dependency {group_id}:{artifact_id} not found in pom.xml"
        
        # Get all available versions
        all_versions = maven_client.get_all_versions(group_id, artifact_id, limit=10)
        latest_version = all_versions[0] if all_versions else None
        
        results = []
        results.append(f"📦 Dependency Analysis: {group_id}:{artifact_id}")
        results.append(f"Current Version: {target_dep.current_version}")
        results.append(f"Latest Version:  {latest_version or 'Unknown'}")
        
        if all_versions:
            results.append(f"\nAvailable Versions (latest 10):")
            for i, ver in enumerate(all_versions[:10]):
                marker = " ← current" if ver == target_dep.current_version else ""
                marker += " ← latest" if ver == latest_version else ""
                results.append(f"  {i+1:2}. {ver}{marker}")
        
        if latest_version and latest_version != target_dep.current_version:
            try:
                if version.parse(latest_version) > version.parse(target_dep.current_version):
                    results.append(f"\n🔄 Update recommended: {target_dep.current_version} → {latest_version}")
                else:
                    results.append(f"\n✅ Current version is up to date")
            except:
                results.append(f"\n⚠️ Could not compare versions")
        
        return "\n".join(results)
        
    except Exception as e:
        return f"❌ Error checking dependency: {str(e)}"

@tool
def update_specific_dependency(project_path: str, group_id: str, artifact_id: str, new_version: str) -> str:
    """Update a specific dependency to a new version."""
    pom_path = os.path.join(project_path, "pom.xml")
    
    if not os.path.exists(pom_path):
        return "❌ pom.xml not found in project"
    
    try:
        parser = PomParser(pom_path)
        
        if parser.update_dependency_version(group_id, artifact_id, new_version):
            if parser.save_pom():
                return f"✅ Successfully updated {group_id}:{artifact_id} to version {new_version}"
            else:
                return f"❌ Updated dependency but failed to save pom.xml"
        else:
            return f"❌ Failed to update {group_id}:{artifact_id} - dependency not found"
            
    except Exception as e:
        return f"❌ Error updating dependency: {str(e)}"

# Updated function with proper Maven Central integration
def _update_deprecated_plugins(project_path: str, lock_manager: FileLockManager) -> List[str]:
    """Update deprecated Maven plugins using Maven Central API."""
    pom_path = os.path.join(project_path, "pom.xml")
    replacements = []
    
    if not lock_manager.acquire_lock(pom_path):
        return ["Error: Could not acquire lock for pom.xml"]
    
    try:
        parser = PomParser(pom_path)
        maven_client = MavenCentralClient()
        
        plugins = parser.get_plugins()
        updates_applied = 0
        
        for plugin in plugins:
            # Check for newer version
            latest_version = maven_client.get_latest_version(plugin.group_id, plugin.artifact_id)
            
            if latest_version and latest_version != plugin.current_version:
                try:
                    if version.parse(latest_version) > version.parse(plugin.current_version):
                        if parser.update_plugin_version(plugin.group_id, plugin.artifact_id, latest_version):
                            replacements.append(f"Updated {plugin.group_id}:{plugin.artifact_id} from {plugin.current_version} to {latest_version}")
                            updates_applied += 1
                except Exception as e:
                    replacements.append(f"Version comparison failed for {plugin.group_id}:{plugin.artifact_id}: {e}")
        
        if updates_applied > 0:
            if parser.save_pom():
                replacements.append(f"Successfully saved {updates_applied} plugin updates to pom.xml")
            else:
                replacements.append("Failed to save updated pom.xml")
        
    except Exception as e:
        replacements.append(f"Error updating plugins: {str(e)}")
    finally:
        lock_manager.release_lock(pom_path)
    
    return replacements

# Advanced Deprecation Detection Tools

@tool
def run_maven_with_deprecation_analysis(project_path: str) -> str:
    """Run Maven build with detailed deprecation warning capture."""
    try:
        # Run Maven with verbose deprecation warnings
        cmd = [
            "mvn", "clean", "compile", 
            "-Xlint:deprecation",
            "-Dmaven.compiler.showDeprecation=true",
            "-Dmaven.compiler.showWarnings=true",
            "-X",  # Debug mode to capture more details
            "-f", project_path
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True, cwd=project_path, timeout=600)
        
        output = result.stdout + result.stderr
        
        # Parse deprecation warnings from output
        deprecations = _parse_maven_deprecations(output)
        
        return f"""
Maven Build Analysis Complete:
Return Code: {result.returncode}

Deprecation Warnings Found: {len(deprecations)}

{chr(10).join(deprecations[:20])}  # Show first 20

Full build output length: {len(output)} characters
        """
        
    except subprocess.TimeoutExpired:
        return "❌ Maven build timed out after 10 minutes"
    except Exception as e:
        return f"❌ Error running Maven build: {str(e)}"

def _parse_maven_deprecations(output: str) -> List[str]:
    """Parse deprecation warnings from Maven output."""
    deprecations = []
    
    # Common deprecation warning patterns
    patterns = [
        r'warning: \[deprecation\] (.*?) is deprecated',
        r'warning: \[deprecation\] (.*?) in (.*?) has been deprecated',
        r'WARNING: (.*?) is deprecated',
        r'DEPRECATED: (.*?)(?=\n)',
        r'@Deprecated.*?public.*?(class|interface|method) (\w+)',
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, output, re.MULTILINE | re.IGNORECASE)
        for match in matches:
            if isinstance(match, tuple):
                deprecations.append(' '.join(str(m) for m in match))
            else:
                deprecations.append(str(match))
    
    return list(set(deprecations))  # Remove duplicates

@tool
def analyze_deprecated_apis_with_llm(project_path: str, deprecation_warnings: str) -> str:
    """Use LLM to analyze deprecated APIs and suggest modern replacements."""
    try:
        # Read some source files for context
        source_files = []
        for root, dirs, files in os.walk(os.path.join(project_path, "src")):
            for file in files[:5]:  # Limit to first 5 files for context
                if file.endswith('.java'):
                    file_path = os.path.join(root, file)
                    try:
                        with open(file_path, 'r') as f:
                            content = f.read()[:2000]  # First 2000 chars
                            source_files.append(f"File: {file}\n{content}")
                    except:
                        continue
        
        analysis_prompt = f"""
        Analyze the following Java project for deprecated APIs, methods, and patterns that need modernization for Java 21.
        
        Maven Deprecation Warnings:
        {deprecation_warnings}
        
        Sample Source Code:
        {chr(10).join(source_files[:3])}
        
        Please provide detailed analysis in this JSON format:
        {{
            "deprecated_items": [
                {{
                    "item_type": "api|method|annotation|pattern",
                    "item_name": "specific name or pattern",
                    "deprecation_reason": "why it's deprecated",
                    "suggested_replacement": "modern alternative",
                    "migration_steps": ["step 1", "step 2"],
                    "confidence_score": 0.0-1.0,
                    "complexity": "simple|moderate|complex",
                    "breaking_change": true|false,
                    "code_example": "example of replacement"
                }}
            ],
            "summary": "overall analysis summary",
            "priority_items": ["most critical items to fix first"]
        }}
        
        Focus on:
        1. Java 8-11 deprecated APIs that have modern alternatives
        2. Spring Framework deprecated patterns
        3. JUnit 4 vs JUnit 5 usage
        4. Deprecated Maven plugins
        5. Old annotation patterns
        6. Legacy Java patterns that can be modernized
        """
        
        response = CLAUDE_SONNET.invoke(analysis_prompt)
        
        # Try to parse as JSON, fallback to text if needed
        try:
            import json
            analysis_result = json.loads(response.content)
            return f"LLM Deprecation Analysis:\n{json.dumps(analysis_result, indent=2)}"
        except:
            return f"LLM Deprecation Analysis:\n{response.content}"
            
    except Exception as e:
        return f"Error in LLM analysis: {str(e)}"

@tool
def scan_maven_plugins_deprecation(project_path: str) -> str:
    """Scan Maven plugins for deprecated versions and suggest updates using Maven Central."""
    pom_path = os.path.join(project_path, "pom.xml")
    
    if not os.path.exists(pom_path):
        return "Error: pom.xml not found"
    
    try:
        parser = PomParser(pom_path)
        maven_client = MavenCentralClient()
        
        plugins = parser.get_plugins()
        deprecated_plugins = []
        
        for plugin in plugins:
            # Check Maven Central for latest version
            latest_version = maven_client.get_latest_version(plugin.group_id, plugin.artifact_id)
            
            if latest_version and latest_version != plugin.current_version:
                try:
                    if version.parse(latest_version) > version.parse(plugin.current_version):
                        deprecated_plugins.append({
                            "plugin": f"{plugin.group_id}:{plugin.artifact_id}",
                            "current_version": plugin.current_version,
                            "recommended_version": latest_version,
                            "notes": "Newer version available on Maven Central"
                        })
                except Exception as e:
                    print(f"Version comparison failed for {plugin.group_id}:{plugin.artifact_id}: {e}")
        
        if deprecated_plugins:
            result = "Outdated Maven Plugins Found:\n"
            for plugin in deprecated_plugins:
                result += f"- {plugin['plugin']} {plugin['current_version']} → {plugin['recommended_version']}\n"
                result += f"  Notes: {plugin['notes']}\n"
            return result
        else:
            return "All Maven plugins are up to date"
            
    except Exception as e:
        return f"Error scanning Maven plugins: {str(e)}"

@tool
def auto_replace_simple_deprecations(project_path: str, deprecation_analysis: str) -> str:
    """Automatically replace simple deprecated patterns with modern alternatives."""
    lock_manager = FileLockManager(project_path)
    replacements_made = []
    
    try:
        # Parse the deprecation analysis to extract actionable items
        simple_replacements = _extract_simple_replacements(deprecation_analysis)
        
        # Apply code-level replacements
        for root, dirs, files in os.walk(os.path.join(project_path, "src")):
            for file in files:
                if file.endswith('.java'):
                    file_path = os.path.join(root, file)
                    
                    if lock_manager.acquire_lock(file_path):
                        try:
                            with open(file_path, 'r') as f:
                                content = f.read()
                            
                            original_content = content
                            
                            # Apply simple replacements
                            for old_pattern, new_pattern, description in simple_replacements:
                                if old_pattern in content:
                                    content = content.replace(old_pattern, new_pattern)
                                    replacements_made.append(f"{file_path}: {description}")
                            
                            # Additional common Java deprecation fixes
                            java_modernizations = {
                                "new Integer(": "Integer.valueOf(",
                                "new Long(": "Long.valueOf(",
                                "new Double(": "Double.valueOf(",
                                "new Boolean(": "Boolean.valueOf(",
                                "new BigDecimal(": "BigDecimal.valueOf(",
                                "Thread.stop()": "// Thread.stop() deprecated - use interrupt()",
                                ".finalize()": "// finalize() deprecated - use try-with-resources",
                                "System.runFinalizersOnExit": "// runFinalizersOnExit deprecated"
                            }
                            
                            for old_api, new_api in java_modernizations.items():
                                if old_api in content:
                                    content = content.replace(old_api, new_api)
                                    replacements_made.append(f"{file_path}: Modernized {old_api}")
                            
                            if content != original_content:
                                with open(file_path, 'w') as f:
                                    f.write(content)
                                    
                        finally:
                            lock_manager.release_lock(file_path)
        
        # Apply POM-level plugin updates
        pom_replacements = _update_deprecated_plugins(project_path, lock_manager)
        replacements_made.extend(pom_replacements)
        
        if replacements_made:
            return f"Automatic replacements applied:\n" + "\n".join(replacements_made)
        else:
            return "No simple deprecations found that could be automatically replaced"
            
    except Exception as e:
        return f"Error applying automatic replacements: {str(e)}"

def _extract_simple_replacements(analysis: str) -> List[tuple]:
    """Extract simple find/replace patterns from LLM analysis."""
    replacements = [
        # Common simple replacements that are safe to automate
        ("@Before", "@BeforeEach", "JUnit 4 to 5 migration"),
        ("@After", "@AfterEach", "JUnit 4 to 5 migration"),
        ("@BeforeClass", "@BeforeAll", "JUnit 4 to 5 migration"),
        ("@AfterClass", "@AfterAll", "JUnit 4 to 5 migration"),
        ("org.junit.Test", "org.junit.jupiter.api.Test", "JUnit 5 test import"),
        ("org.junit.Assert", "org.junit.jupiter.api.Assertions", "JUnit 5 assertions"),
        ("javax.annotation.PostConstruct", "jakarta.annotation.PostConstruct", "Jakarta migration"),
        ("javax.annotation.PreDestroy", "jakarta.annotation.PreDestroy", "Jakarta migration"),
    ]
    
    return replacements

@tool 
def generate_deprecation_report(project_path: str, analysis_results: str) -> str:
    """Generate comprehensive deprecation analysis report."""
    try:
        report_content = f"""
# Deprecation Analysis Report

**Project**: {project_path}
**Analysis Date**: {datetime.now().isoformat()}

## Executive Summary
{analysis_results[:500]}...

## Detailed Findings

### Deprecated APIs and Methods
- Items requiring immediate attention
- Breaking changes that need careful migration
- Simple replacements that can be automated

### Maven Plugin Updates
- Outdated plugins affecting build process
- Security and compatibility improvements available

### Recommended Migration Priority
1. **High Priority**: Breaking changes and security issues
2. **Medium Priority**: Performance improvements and better APIs
3. **Low Priority**: Code style and minor optimizations

## Next Steps
1. Review all suggested changes
2. Test automated replacements in development environment
3. Plan manual migration for complex items
4. Update CI/CD pipeline configurations

---
*Generated by Enhanced Java Migration System*
        """
        
        report_path = os.path.join(project_path, "deprecation-analysis-report.md")
        with open(report_path, 'w') as f:
            f.write(report_content)
        
        return f"Deprecation report generated: {report_path}"
        
    except Exception as e:
        return f"Error generating report: {str(e)}"

print("✅ Enhanced dependency management with Maven Central integration implemented!")

✅ Enhanced dependency management with Maven Central integration implemented!


In [57]:
# Migration Phase Configuration and Deprecation Detection

from enum import Enum
from dataclasses import dataclass
from typing import Set, Dict, List, Optional

class MigrationPhaseType(Enum):
    """All available migration phases."""
    ANALYSIS = "analysis"
    DEPRECATION_DETECTION = "deprecation_detection"  # NEW PHASE!
    DEPENDENCY_UPDATE = "dependency_update"
    CODE_MIGRATION = "code_migration"
    TESTING_VALIDATION = "testing_validation"
    PERFORMANCE_VALIDATION = "performance_validation"
    FINAL_CLEANUP = "final_cleanup"

@dataclass
class PhaseConfig:
    """Configuration for individual migration phases."""
    enabled: bool = True
    priority: int = 1
    timeout_minutes: int = 30
    retry_count: int = 3
    skip_on_failure: bool = False
    custom_params: Dict[str, any] = None

@dataclass
class DeprecationItem:
    """Represents a deprecated item found in the project."""
    item_type: str  # 'api', 'plugin', 'dependency', 'annotation', 'method'
    item_name: str
    location: str  # file path or configuration location
    current_usage: str  # current code/config
    deprecation_reason: str
    suggested_replacement: str
    confidence_score: float  # 0.0 to 1.0
    migration_complexity: str  # 'simple', 'moderate', 'complex'
    breaking_change: bool
    additional_notes: str

class MigrationConfiguration:
    """Configurable migration phases and settings."""
    
    def __init__(self):
        self.phases = {
            MigrationPhaseType.ANALYSIS: PhaseConfig(enabled=True, priority=1),
            MigrationPhaseType.DEPRECATION_DETECTION: PhaseConfig(
                enabled=True, 
                priority=2,
                timeout_minutes=45,
                custom_params={
                    "deep_scan": True,
                    "include_transitive_deps": True,
                    "check_plugin_compatibility": True,
                    "analyze_code_patterns": True
                }
            ),
            MigrationPhaseType.DEPENDENCY_UPDATE: PhaseConfig(enabled=True, priority=3),
            MigrationPhaseType.CODE_MIGRATION: PhaseConfig(enabled=True, priority=4),
            MigrationPhaseType.TESTING_VALIDATION: PhaseConfig(enabled=True, priority=5),
            MigrationPhaseType.PERFORMANCE_VALIDATION: PhaseConfig(enabled=False, priority=6),
            MigrationPhaseType.FINAL_CLEANUP: PhaseConfig(enabled=True, priority=7)
        }
        
        self.global_settings = {
            "auto_apply_safe_changes": True,
            "require_confirmation_for_breaking_changes": True,
            "create_backup_before_changes": True,
            "max_parallel_operations": 3,
            "llm_analysis_model": "claude-sonnet",
            "web_research_enabled": True
        }
    
    def enable_phase(self, phase: MigrationPhaseType, **kwargs):
        """Enable a migration phase with optional configuration."""
        if phase in self.phases:
            self.phases[phase].enabled = True
            for key, value in kwargs.items():
                setattr(self.phases[phase], key, value)
    
    def disable_phase(self, phase: MigrationPhaseType):
        """Disable a migration phase."""
        if phase in self.phases:
            self.phases[phase].enabled = False
    
    def get_enabled_phases(self) -> List[MigrationPhaseType]:
        """Get all enabled phases in priority order."""
        enabled = [(phase, config) for phase, config in self.phases.items() if config.enabled]
        return [phase for phase, config in sorted(enabled, key=lambda x: x[1].priority)]
    
    def update_phase_config(self, phase: MigrationPhaseType, **kwargs):
        """Update configuration for a specific phase."""
        if phase in self.phases:
            for key, value in kwargs.items():
                if hasattr(self.phases[phase], key):
                    setattr(self.phases[phase], key, value)
                elif self.phases[phase].custom_params is None:
                    self.phases[phase].custom_params = {key: value}
                else:
                    self.phases[phase].custom_params[key] = value

print("✅ Migration phase configuration system implemented!")

✅ Migration phase configuration system implemented!


In [ ]:
# Build Error Resolution and Web Research Tools with Intelligent Error Fixing

@tool
def analyze_build_errors_with_ai(project_path: str, build_output: str) -> str:
    """Use AI to analyze build errors and suggest fixes."""
    try:
        analysis_prompt = f"""
        Analyze the following Maven build errors and provide specific fix suggestions:
        
        Build Output:
        {build_output[:2000]}  # Limit context
        
        Please provide:
        1. Root cause analysis
        2. Specific code or configuration changes needed
        3. Priority level (high/medium/low)
        4. Whether this requires dependency updates
        """
        
        response = CLAUDE_SONNET.invoke(analysis_prompt)
        return f"AI Build Error Analysis:\n{response.content}"
        
    except Exception as e:
        return f"Error in AI analysis: {str(e)}"

@tool
def run_build_fix_cycle(project_path: str, max_iterations: int = 3) -> str:
    """Run build, analyze errors, apply fixes, and retry up to max_iterations with intelligent error fixing."""
    results = []
    
    for iteration in range(max_iterations):
        results.append(f"\n--- Build Iteration {iteration + 1} ---")
        
        # Run build
        build_result = run_maven_build(project_path)
        results.append(f"Build Result: {build_result[:500]}")
        
        if "✅ Build completed successfully" in build_result:
            results.append("🎉 Build is now successful!")
            break
        
        # Analyze build errors with AI
        ai_analysis = analyze_build_errors_with_ai(project_path, build_result)
        results.append(f"AI Analysis: {ai_analysis[:300]}")
        
        # Apply automatic fixes based on error type
        auto_fixes = auto_fix_build_errors(project_path, build_result)
        results.append(f"Auto Fixes: {auto_fixes}")
        
        # If no automatic fixes were applied, try intelligent error fixing
        if "No automatic fixes" in auto_fixes or "Error applying build fixes" in auto_fixes:
            results.append("🧠 Trying intelligent error fixing for unknown errors...")
            
            # Extract build errors and apply intelligent fixing
            build_errors = build_result.split('\n')
            error_lines = [line for line in build_errors if 'ERROR' in line.upper() or 'FAILED' in line.upper()]
            
            if error_lines:
                # Try intelligent error fixing on the first few critical errors
                for error_line in error_lines[:3]:  # Limit to first 3 errors to avoid overwhelming
                    intelligent_fix_result = intelligent_error_fixer(error_line)
                    results.append(f"Intelligent Fix: {intelligent_fix_result[:200]}...")
            else:
                # If no specific errors found, try web research
                web_research = search_build_error_solutions(build_result)
                results.append(f"Web Research: {web_research[:200]}")
                break
    
    return "\n".join(results)

@tool
def auto_fix_build_errors(project_path: str, build_output: str) -> str:
    """Automatically fix common build errors."""
    lock_manager = FileLockManager(project_path)
    fixes_applied = []
    
    try:
        # Fix missing package errors
        if "package does not exist" in build_output.lower():
            fixes_applied.extend(_fix_missing_imports(project_path, lock_manager))
        
        # Fix deprecated API usage
        if "deprecated" in build_output.lower():
            fixes_applied.extend(_fix_deprecated_apis(project_path, lock_manager))
        
        # Fix missing dependencies
        if "cannot find symbol" in build_output.lower():
            fixes_applied.extend(_suggest_missing_dependencies(project_path, build_output))
        
        if fixes_applied:
            return f"Applied build fixes:\n" + "\n".join(fixes_applied)
        else:
            return "No automatic build fixes available for this error type"
            
    except Exception as e:
        return f"Error applying build fixes: {str(e)}"

def _fix_deprecated_apis(project_path: str, lock_manager: FileLockManager) -> List[str]:
    """Fix common deprecated API usage."""
    fixes = []
    
    # Common deprecated API fixes for Java 21
    api_fixes = {
        "new Integer(": "Integer.valueOf(",
        "new Long(": "Long.valueOf(",
        "new Double(": "Double.valueOf(",
        "new Boolean(": "Boolean.valueOf(",
    }
    
    for root, dirs, files in os.walk(os.path.join(project_path, "src")):
        for file in files:
            if file.endswith('.java'):
                file_path = os.path.join(root, file)
                
                if lock_manager.acquire_lock(file_path):
                    try:
                        with open(file_path, 'r') as f:
                            content = f.read()
                        
                        original_content = content
                        for old_api, new_api in api_fixes.items():
                            if old_api in content:
                                content = content.replace(old_api, new_api)
                        
                        if content != original_content:
                            with open(file_path, 'w') as f:
                                f.write(content)
                            fixes.append(f"Fixed deprecated APIs in {file_path}")
                            
                    finally:
                        lock_manager.release_lock(file_path)
    
    return fixes

def _suggest_missing_dependencies(project_path: str, build_output: str) -> List[str]:
    """Suggest missing dependencies based on build errors."""
    suggestions = []
    
    # Common missing dependencies for Java 21 migration
    dependency_mappings = {
        "jakarta.servlet": "jakarta.servlet:jakarta.servlet-api",
        "jakarta.persistence": "jakarta.persistence:jakarta.persistence-api",
        "org.junit.jupiter": "org.junit.jupiter:junit-jupiter",
    }
    
    for symbol, dependency in dependency_mappings.items():
        if symbol in build_output:
            suggestions.append(f"Consider adding dependency: {dependency}")
    
    return suggestions

# Web Research Integration with Bing Search API

@tool
def search_build_error_solutions(error_message: str) -> str:
    """Search for build error solutions using Bing Search API."""
    if not BING_SEARCH_API_KEY or BING_SEARCH_API_KEY == 'your-bing-api-key':
        return "Bing Search API key not configured"
    
    try:
        # Extract key error terms
        error_terms = _extract_error_terms(error_message)
        query = f"java maven build error {error_terms} solution"
        
        headers = {'Ocp-Apim-Subscription-Key': BING_SEARCH_API_KEY}
        params = {
            'q': query,
            'count': 5,
            'mkt': 'en-US',
            'safeSearch': 'Moderate'
        }
        
        response = requests.get(
            'https://api.bing.microsoft.com/v7.0/search',
            headers=headers,
            params=params,
            timeout=10
        )
        response.raise_for_status()
        
        results = response.json()
        search_results = []
        
        for item in results.get('webPages', {}).get('value', [])[:3]:
            title = item.get('name', '')
            url = item.get('url', '')
            snippet = item.get('snippet', '')
            search_results.append(f"Title: {title}\nURL: {url}\nSnippet: {snippet}\n")
        
        if search_results:
            return "Web search results:\n" + "\n---\n".join(search_results)
        else:
            return "No relevant search results found"
            
    except Exception as e:
        return f"Error searching for solutions: {str(e)}"

def _extract_error_terms(error_message: str) -> str:
    """Extract key terms from error message for search."""
    # Common error patterns to extract
    patterns = [
        r'package ([\w.]+) does not exist',
        r'cannot find symbol.*?symbol:\s*(\w+)',
        r'class (\w+) is public, should be declared in a file named',
        r'(\w+Exception): (.+?)(?=\n|\Z)'
    ]
    
    terms = []
    for pattern in patterns:
        matches = re.findall(pattern, error_message)
        for match in matches:
            if isinstance(match, tuple):
                terms.extend([str(m) for m in match if str(m).strip()])
            else:
                terms.append(str(match).strip())
    
    # Clean and limit terms
    clean_terms = [term for term in terms if len(term) > 2 and len(term) < 50]
    return " ".join(clean_terms[:5])

@tool
def research_migration_patterns(topic: str) -> str:
    """Research specific migration patterns using web search."""
    if not BING_SEARCH_API_KEY or BING_SEARCH_API_KEY == 'your-bing-api-key':
        return "Bing Search API key not configured"
    
    try:
        query = f"java 11 to 21 migration {topic} best practices"
        
        # Use Claude Haiku for web research as specified
        research_results = _perform_web_search(query)
        
        analysis_prompt = f"""
        Research the following migration topic based on web search results:
        
        Topic: {topic}
        Search Results: {research_results[:1500]}
        
        Please provide:
        1. Key migration strategies
        2. Common pitfalls to avoid
        3. Recommended tools or libraries
        4. Code examples if applicable
        """
        
        response = CLAUDE_HAIKU.invoke(analysis_prompt)
        return f"Migration Research for '{topic}':\n{response.content}"
        
    except Exception as e:
        return f"Error researching migration patterns: {str(e)}"

def _perform_web_search(query: str) -> str:
    """Perform web search and return formatted results."""
    try:
        headers = {'Ocp-Apim-Subscription-Key': BING_SEARCH_API_KEY}
        params = {
            'q': query,
            'count': 5,
            'mkt': 'en-US',
            'safeSearch': 'Moderate'
        }
        
        response = requests.get(
            'https://api.bing.microsoft.com/v7.0/search',
            headers=headers,
            params=params,
            timeout=10
        )
        response.raise_for_status()
        
        results = response.json()
        formatted_results = []
        
        for item in results.get('webPages', {}).get('value', []):
            title = item.get('name', '')
            snippet = item.get('snippet', '')
            formatted_results.append(f"{title}: {snippet}")
        
        return "\n".join(formatted_results)
        
    except Exception as e:
        return f"Search error: {str(e)}"

print("✅ Build error resolution and web research tools with intelligent error fixing implemented successfully!")

✅ Build error resolution and web research tools with intelligent error fixing implemented successfully!


In [59]:
# Test Analysis and Fixing Tools with Retry Loops and Intelligent Error Fixing

@tool
def analyze_test_failures_with_ai(project_path: str, test_output: str) -> str:
    """Use AI to analyze test failures and suggest fixes."""
    try:
        analysis_prompt = f"""
        Analyze the following Maven test failures and provide specific fix suggestions:
        
        Test Output:
        {test_output[:2000]}  # Limit context
        
        Please provide:
        1. Root cause analysis
        2. Specific code changes needed
        3. Priority level (high/medium/low)
        4. Estimated fix complexity
        """
        
        response = CLAUDE_SONNET.invoke(analysis_prompt)
        return f"AI Test Failure Analysis:\n{response.content}"
        
    except Exception as e:
        return f"Error in AI analysis: {str(e)}"

@tool
def auto_fix_common_test_issues(project_path: str, failure_type: str) -> str:
    """Automatically fix common test issues based on failure type."""
    lock_manager = FileLockManager(project_path)
    fixes_applied = []
    
    try:
        if "package does not exist" in failure_type.lower():
            fixes_applied.extend(_fix_missing_imports(project_path, lock_manager))
        
        if "assertionerror" in failure_type.lower():
            fixes_applied.extend(_fix_assertion_issues(project_path, lock_manager))
        
        if "classnotfoundexception" in failure_type.lower():
            fixes_applied.extend(_fix_missing_classes(project_path, lock_manager))
        
        if fixes_applied:
            return f"Applied fixes:\n" + "\n".join(fixes_applied)
        else:
            return "No automatic fixes available for this issue type"
            
    except Exception as e:
        return f"Error applying automatic fixes: {str(e)}"

def _fix_missing_imports(project_path: str, lock_manager: FileLockManager) -> List[str]:
    """Fix missing import statements."""
    fixes = []
    
    # Common import fixes for Java 21 migration
    import_fixes = {
        "javax.servlet": "jakarta.servlet",
        "javax.persistence": "jakarta.persistence",
        "javax.validation": "jakarta.validation"
    }
    
    for root, dirs, files in os.walk(os.path.join(project_path, "src")):
        for file in files:
            if file.endswith('.java'):
                file_path = os.path.join(root, file)
                
                if lock_manager.acquire_lock(file_path):
                    try:
                        with open(file_path, 'r') as f:
                            content = f.read()
                        
                        original_content = content
                        for old_import, new_import in import_fixes.items():
                            if f"import {old_import}" in content:
                                content = content.replace(f"import {old_import}", f"import {new_import}")
                        
                        if content != original_content:
                            with open(file_path, 'w') as f:
                                f.write(content)
                            fixes.append(f"Updated imports in {file_path}")
                            
                    finally:
                        lock_manager.release_lock(file_path)
    
    return fixes

def _fix_assertion_issues(project_path: str, lock_manager: FileLockManager) -> List[str]:
    """Fix common assertion issues in tests."""
    fixes = []
    
    # JUnit 4 to 5 assertion fixes
    assertion_fixes = {
        "Assert.assertEquals": "Assertions.assertEquals",
        "Assert.assertTrue": "Assertions.assertTrue",
        "Assert.assertFalse": "Assertions.assertFalse",
        "Assert.assertNull": "Assertions.assertNull",
        "Assert.assertNotNull": "Assertions.assertNotNull"
    }
    
    test_dirs = [
        os.path.join(project_path, "src", "test"),
        os.path.join(project_path, "test")
    ]
    
    for test_dir in test_dirs:
        if os.path.exists(test_dir):
            for root, dirs, files in os.walk(test_dir):
                for file in files:
                    if file.endswith('.java'):
                        file_path = os.path.join(root, file)
                        
                        if lock_manager.acquire_lock(file_path):
                            try:
                                with open(file_path, 'r') as f:
                                    content = f.read()
                                
                                original_content = content
                                for old_assertion, new_assertion in assertion_fixes.items():
                                    content = content.replace(old_assertion, new_assertion)
                                
                                # Add JUnit 5 import if needed
                                if content != original_content and "import org.junit.jupiter.api.Assertions" not in content:
                                    content = "import org.junit.jupiter.api.Assertions;\n" + content
                                
                                if content != original_content:
                                    with open(file_path, 'w') as f:
                                        f.write(content)
                                    fixes.append(f"Fixed assertions in {file_path}")
                                    
                            finally:
                                lock_manager.release_lock(file_path)
    
    return fixes

def _fix_missing_classes(project_path: str, lock_manager: FileLockManager) -> List[str]:
    """Fix missing class references."""
    # This would implement more complex class resolution logic
    return ["Missing class fixes not yet implemented"]

@tool
def run_test_fix_cycle(project_path: str, max_iterations: int = 3) -> str:
    """Run tests, analyze failures, apply fixes, and retry up to max_iterations with intelligent error fixing."""
    results = []
    
    for iteration in range(max_iterations):
        results.append(f"\n--- Test Iteration {iteration + 1} ---")
        
        # Run tests
        test_result = run_maven_tests(project_path)
        results.append(f"Test Result: {test_result[:500]}")
        
        if "✅ Tests passed successfully" in test_result:
            results.append("🎉 All tests are now passing!")
            break
        
        # Analyze failures with AI
        ai_analysis = analyze_test_failures_with_ai(project_path, test_result)
        results.append(f"AI Analysis: {ai_analysis[:300]}")
        
        # Apply automatic fixes
        auto_fixes = auto_fix_common_test_issues(project_path, test_result)
        results.append(f"Auto Fixes: {auto_fixes}")
        
        # If no automatic fixes were applied, try intelligent error fixing
        if "No automatic fixes" in auto_fixes or "Error applying automatic fixes" in auto_fixes:
            results.append("🧠 Trying intelligent error fixing for unknown test failures...")
            
            # Extract test failures and apply intelligent fixing
            test_failures = test_result.split('\n')
            failure_lines = [line for line in test_failures if 'FAILURE' in line.upper() or 'ERROR' in line.upper() or 'failed' in line.lower()]
            
            if failure_lines:
                # Try intelligent error fixing on the first few critical failures
                for failure_line in failure_lines[:3]:  # Limit to first 3 failures
                    intelligent_fix_result = intelligent_error_fixer(failure_line)
                    results.append(f"Intelligent Test Fix: {intelligent_fix_result[:200]}...")
            else:
                results.append("⚠️ No more automatic or intelligent fixes available")
                break
    
    return "\n".join(results)

print("✅ Test analysis and fixing tools with intelligent error fixing implemented successfully!")

✅ Test analysis and fixing tools with intelligent error fixing implemented successfully!


In [60]:
# File Lock Manager for Concurrent Operations
class FileLockManager:
    """Manages file-level locking for concurrent modifications."""
    
    def __init__(self, base_path: str):
        self.base_path = Path(base_path)
        self.locks: Dict[str, FileLock] = {}
        self.lock_registry = threading.Lock()
    
    def acquire_lock(self, file_path: str, timeout: int = 30) -> bool:
        """Acquire a lock for a specific file."""
        normalized_path = str(Path(file_path).resolve())
        lock_file = self.base_path / f".locks/{hash(normalized_path)}.lock"
        
        with self.lock_registry:
            if normalized_path not in self.locks:
                lock_file.parent.mkdir(exist_ok=True)
                self.locks[normalized_path] = FileLock(str(lock_file))
        
        try:
            self.locks[normalized_path].acquire(timeout=timeout)
            return True
        except:
            return False
    
    def release_lock(self, file_path: str):
        """Release a lock for a specific file."""
        normalized_path = str(Path(file_path).resolve())
        
        with self.lock_registry:
            if normalized_path in self.locks:
                try:
                    self.locks[normalized_path].release()
                except:
                    pass
    
    def is_locked(self, file_path: str) -> bool:
        """Check if a file is currently locked."""
        normalized_path = str(Path(file_path).resolve())
        
        with self.lock_registry:
            if normalized_path in self.locks:
                return self.locks[normalized_path].is_locked
        return False

print("✅ FileLockManager implemented successfully!")

✅ FileLockManager implemented successfully!


In [61]:
# Enhanced File Operations and Intelligent Error Fixing Tools

@tool
def read_file_content(file_path: str) -> str:
    """Read the complete content of a file."""
    try:
        if not os.path.exists(file_path):
            return f"❌ File not found: {file_path}"
        
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        return f"✅ File content from {file_path}:\n{content}"
        
    except Exception as e:
        return f"❌ Error reading file {file_path}: {str(e)}"

@tool 
def write_file_content(file_path: str, content: str) -> str:
    """Write content to a file, creating directories if needed."""
    try:
        # Create directories if they don't exist
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(content)
        
        return f"✅ Successfully wrote {len(content)} characters to {file_path}"
        
    except Exception as e:
        return f"❌ Error writing to file {file_path}: {str(e)}"

@tool
def modify_file_content(file_path: str, old_text: str, new_text: str, all_occurrences: bool = False) -> str:
    """Modify specific text in a file with find and replace."""
    lock_manager = FileLockManager(os.path.dirname(file_path))
    
    if not lock_manager.acquire_lock(file_path):
        return f"❌ Could not acquire lock for {file_path}"
    
    try:
        if not os.path.exists(file_path):
            return f"❌ File not found: {file_path}"
        
        # Read current content
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        original_content = content
        
        # Perform replacement
        if all_occurrences:
            content = content.replace(old_text, new_text)
            replacement_count = original_content.count(old_text)
        else:
            if old_text in content:
                content = content.replace(old_text, new_text, 1)
                replacement_count = 1
            else:
                replacement_count = 0
        
        if replacement_count == 0:
            return f"⚠️ Text not found in {file_path}: '{old_text[:100]}...'"
        
        # Write modified content back
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(content)
        
        return f"✅ Successfully replaced {replacement_count} occurrence(s) in {file_path}"
        
    except Exception as e:
        return f"❌ Error modifying file {file_path}: {str(e)}"
    finally:
        lock_manager.release_lock(file_path)

@tool
def insert_text_at_line(file_path: str, line_number: int, text_to_insert: str) -> str:
    """Insert text at a specific line number in a file."""
    lock_manager = FileLockManager(os.path.dirname(file_path))
    
    if not lock_manager.acquire_lock(file_path):
        return f"❌ Could not acquire lock for {file_path}"
    
    try:
        if not os.path.exists(file_path):
            return f"❌ File not found: {file_path}"
        
        # Read all lines
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        
        # Insert text at specified line (1-based indexing)
        if line_number < 1 or line_number > len(lines) + 1:
            return f"❌ Invalid line number {line_number}. File has {len(lines)} lines."
        
        # Ensure text ends with newline if it doesn't already
        if not text_to_insert.endswith('\n'):
            text_to_insert += '\n'
        
        lines.insert(line_number - 1, text_to_insert)
        
        # Write back to file
        with open(file_path, 'w', encoding='utf-8') as f:
            f.writelines(lines)
        
        return f"✅ Successfully inserted text at line {line_number} in {file_path}"
        
    except Exception as e:
        return f"❌ Error inserting text in file {file_path}: {str(e)}"
    finally:
        lock_manager.release_lock(file_path)

@tool
def remove_lines_from_file(file_path: str, start_line: int, end_line: int = None) -> str:
    """Remove line(s) from a file. If end_line not specified, removes only start_line."""
    lock_manager = FileLockManager(os.path.dirname(file_path))
    
    if not lock_manager.acquire_lock(file_path):
        return f"❌ Could not acquire lock for {file_path}"
    
    try:
        if not os.path.exists(file_path):
            return f"❌ File not found: {file_path}"
        
        # Read all lines
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        
        if end_line is None:
            end_line = start_line
        
        # Validate line numbers (1-based indexing)
        if start_line < 1 or start_line > len(lines):
            return f"❌ Invalid start line {start_line}. File has {len(lines)} lines."
        
        if end_line < start_line or end_line > len(lines):
            return f"❌ Invalid end line {end_line}. Must be >= start_line and <= {len(lines)}."
        
        # Remove lines (convert to 0-based indexing)
        removed_lines = lines[start_line-1:end_line]
        del lines[start_line-1:end_line]
        
        # Write back to file
        with open(file_path, 'w', encoding='utf-8') as f:
            f.writelines(lines)
        
        lines_removed = end_line - start_line + 1
        return f"✅ Successfully removed {lines_removed} line(s) from {file_path}"
        
    except Exception as e:
        return f"❌ Error removing lines from file {file_path}: {str(e)}"
    finally:
        lock_manager.release_lock(file_path)

@tool
def add_import_statement(file_path: str, import_statement: str) -> str:
    """Add an import statement to a Java file, placing it in the correct location."""
    lock_manager = FileLockManager(os.path.dirname(file_path))
    
    if not lock_manager.acquire_lock(file_path):
        return f"❌ Could not acquire lock for {file_path}"
    
    try:
        if not os.path.exists(file_path):
            return f"❌ File not found: {file_path}"
        
        # Read file content
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        
        # Ensure import statement has proper format
        if not import_statement.startswith('import '):
            import_statement = f"import {import_statement}"
        if not import_statement.endswith(';\n'):
            import_statement = import_statement.rstrip() + ';\n'
        
        # Check if import already exists
        import_without_semicolon = import_statement.rstrip(';\n')
        for line in lines:
            if line.strip().rstrip(';') == import_without_semicolon.strip():
                return f"⚠️ Import already exists in {file_path}: {import_statement.strip()}"
        
        # Find the correct position to insert import
        insert_position = 0
        package_line_found = False
        
        for i, line in enumerate(lines):
            if line.strip().startswith('package '):
                package_line_found = True
                continue
            elif line.strip().startswith('import '):
                continue
            elif package_line_found and line.strip() == '':
                continue
            else:
                # First non-import, non-package, non-empty line
                insert_position = i
                break
        
        # If no package line found, insert at the beginning
        if not package_line_found:
            insert_position = 0
        
        # Insert the import statement
        lines.insert(insert_position, import_statement)
        
        # Write back to file
        with open(file_path, 'w', encoding='utf-8') as f:
            f.writelines(lines)
        
        return f"✅ Successfully added import to {file_path}: {import_statement.strip()}"
        
    except Exception as e:
        return f"❌ Error adding import to file {file_path}: {str(e)}"
    finally:
        lock_manager.release_lock(file_path)

@tool
def intelligent_error_fixer(error_message: str, context_lines_above: int = 40, context_lines_below: int = 50) -> str:
    """
    Intelligently fix unknown errors by analyzing file context around the error location.
    
    This tool:
    1. Parses the error message to extract file path and line number
    2. Reads the file content around the error location 
    3. Sends the error + context to LLM for analysis
    4. Applies the suggested fix to the file
    """
    try:
        # Parse error message to extract file path and line number
        error_info = _parse_error_location(error_message)
        
        if not error_info:
            return f"❌ Could not parse file location from error: {error_message[:200]}"
        
        file_path = error_info['file_path']
        error_line = error_info['line_number']
        error_description = error_info['description']
        
        if not os.path.exists(file_path):
            return f"❌ Error file not found: {file_path}"
        
        # Read file content with context around the error
        context_info = _read_file_context(file_path, error_line, context_lines_above, context_lines_below)
        
        if not context_info:
            return f"❌ Could not read context from file {file_path} around line {error_line}"
        
        # Prepare comprehensive prompt for LLM analysis
        analysis_prompt = f"""
        JAVA COMPILATION ERROR ANALYSIS AND FIX
        
        **Error Details:**
        File: {file_path}
        Line: {error_line}
        Error: {error_description}
        
        **Full Error Message:**
        {error_message}
        
        **File Context (Lines {context_info['start_line']}-{context_info['end_line']}):**
        ```java
        {context_info['context_content']}
        ```
        
        **ERROR LINE (Line {error_line}):**
        ```java
        {context_info['error_line_content']}
        ```
        
        **Analysis Request:**
        Please analyze this Java compilation error and provide a specific fix.
        
        **Required Response Format:**
        {{
            "error_analysis": "Detailed explanation of what's causing the error",
            "fix_type": "import_missing|code_replacement|line_insertion|line_removal|complex_refactor",
            "specific_fix": {{
                "action": "replace|insert|remove|add_import",
                "old_code": "exact code to replace (if action is replace)",
                "new_code": "new code to insert/replace with",
                "target_line": "line number for insertion/removal (if applicable)",
                "import_statement": "import to add (if action is add_import)"
            }},
            "explanation": "Why this fix resolves the error",
            "confidence": 0.95
        }}
        
        **Important Guidelines:**
        1. Focus on the EXACT error at line {error_line}
        2. Consider the surrounding context for proper fix
        3. For Java 21 migration, prefer modern APIs
        4. Ensure the fix maintains code functionality
        5. Be specific - provide exact text replacements
        """
        
        # Get LLM analysis using Claude Sonnet
        try:
            response = CLAUDE_SONNET.invoke(analysis_prompt)
            analysis_result = response.content
            
            # Parse the LLM response
            fix_data = _parse_llm_fix_response(analysis_result)
            
            if not fix_data:
                return f"❌ Could not parse LLM fix response for {file_path}:{error_line}"
            
            # Apply the suggested fix
            fix_result = _apply_intelligent_fix(file_path, fix_data, context_info)
            
            return f"""✅ Intelligent Error Fix Applied:

**Error:** {error_description}
**File:** {file_path}:{error_line}

**Analysis:** {fix_data.get('error_analysis', 'LLM analysis')}

**Fix Applied:** {fix_data.get('explanation', 'Applied suggested fix')}

**Result:** {fix_result}

**Confidence:** {fix_data.get('confidence', 'N/A')}
"""
            
        except Exception as e:
            return f"❌ Error in LLM analysis: {str(e)}"
        
    except Exception as e:
        return f"❌ Error in intelligent error fixing: {str(e)}"

def _parse_error_location(error_message: str) -> Optional[Dict[str, Any]]:
    """Parse error message to extract file path, line number, and description."""
    import re
    
    # Common Java error patterns
    patterns = [
        # Standard Maven compiler errors: [ERROR] /path/to/File.java:[line,col] error description
        r'\[ERROR\]\s+([^:]+\.java):\[(\d+),\d+\]\s*(.*?)(?=\n|\Z)',
        
        # Gradle-style errors: /path/to/File.java:line: error: description
        r'([^:]+\.java):(\d+):\s*error:\s*(.*?)(?=\n|\Z)',
        
        # Simple format: File.java:line error description  
        r'([^:]+\.java):(\d+)\s*(.*?)(?=\n|\Z)',
        
        # Maven test errors: Failed: TestClass.java:line
        r'Failed:\s+([^:]+\.java):(\d+)\s*(.*?)(?=\n|\Z)',
    ]
    
    for pattern in patterns:
        match = re.search(pattern, error_message, re.MULTILINE | re.IGNORECASE)
        if match:
            file_path = match.group(1).strip()
            line_number = int(match.group(2))
            description = match.group(3).strip() if len(match.groups()) > 2 else "Compilation error"
            
            # Convert relative paths to absolute if needed
            if not os.path.isabs(file_path):
                # Try to find the file in common locations
                possible_paths = [
                    file_path,
                    os.path.abspath(file_path),
                ]
                
                for path in possible_paths:
                    if os.path.exists(path):
                        file_path = path
                        break
            
            return {
                'file_path': file_path,
                'line_number': line_number,
                'description': description
            }
    
    return None

def _read_file_context(file_path: str, error_line: int, lines_above: int = 40, lines_below: int = 50) -> Optional[Dict[str, Any]]:
    """Read file content with context around the error line."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            all_lines = f.readlines()
        
        total_lines = len(all_lines)
        
        # Calculate context boundaries
        start_line = max(1, error_line - lines_above)
        end_line = min(total_lines, error_line + lines_below)
        
        # Extract context lines (convert to 0-based indexing)
        context_lines = all_lines[start_line-1:end_line]
        
        # Get the specific error line content
        error_line_content = all_lines[error_line-1].strip() if 1 <= error_line <= total_lines else ""
        
        # Create numbered context content
        context_content = ""
        for i, line in enumerate(context_lines):
            line_num = start_line + i
            marker = " >>> " if line_num == error_line else "     "
            context_content += f"{line_num:4d}{marker}{line.rstrip()}\n"
        
        return {
            'start_line': start_line,
            'end_line': end_line,
            'error_line_content': error_line_content,
            'context_content': context_content,
            'total_lines': total_lines
        }
        
    except Exception as e:
        print(f"Error reading file context: {e}")
        return None

def _parse_llm_fix_response(response_content: str) -> Optional[Dict[str, Any]]:
    """Parse LLM response to extract fix instructions."""
    import json
    import re
    
    try:
        # Try to extract JSON from the response
        json_match = re.search(r'\{.*\}', response_content, re.DOTALL)
        if json_match:
            json_str = json_match.group(0)
            return json.loads(json_str)
        
        # Fallback: parse structured text response
        fix_data = {
            'error_analysis': '',
            'fix_type': 'code_replacement',
            'specific_fix': {},
            'explanation': '',
            'confidence': 0.5
        }
        
        # Extract key information using regex
        analysis_match = re.search(r'error_analysis["\']?\s*:\s*["\']?(.*?)["\']?(?=\n|,|\})', response_content, re.IGNORECASE)
        if analysis_match:
            fix_data['error_analysis'] = analysis_match.group(1).strip()
        
        explanation_match = re.search(r'explanation["\']?\s*:\s*["\']?(.*?)["\']?(?=\n|,|\})', response_content, re.IGNORECASE)
        if explanation_match:
            fix_data['explanation'] = explanation_match.group(1).strip()
        
        # Look for code replacements
        old_code_match = re.search(r'old_code["\']?\s*:\s*["\']?(.*?)["\']?(?=\n|,|\})', response_content, re.DOTALL)
        new_code_match = re.search(r'new_code["\']?\s*:\s*["\']?(.*?)["\']?(?=\n|,|\})', response_content, re.DOTALL)
        
        if old_code_match and new_code_match:
            fix_data['specific_fix'] = {
                'action': 'replace',
                'old_code': old_code_match.group(1).strip(),
                'new_code': new_code_match.group(1).strip()
            }
        
        return fix_data
        
    except Exception as e:
        print(f"Error parsing LLM response: {e}")
        return None

def _apply_intelligent_fix(file_path: str, fix_data: Dict[str, Any], context_info: Dict[str, Any]) -> str:
    """Apply the intelligent fix suggested by the LLM."""
    lock_manager = FileLockManager(os.path.dirname(file_path))
    
    if not lock_manager.acquire_lock(file_path):
        return "❌ Could not acquire file lock"
    
    try:
        specific_fix = fix_data.get('specific_fix', {})
        action = specific_fix.get('action', 'replace')
        
        if action == 'replace':
            old_code = specific_fix.get('old_code', '').strip()
            new_code = specific_fix.get('new_code', '').strip()
            
            if old_code and new_code:
                result = modify_file_content(file_path, old_code, new_code, all_occurrences=False)
                return f"Code replacement: {result}"
        
        elif action == 'add_import':
            import_statement = specific_fix.get('import_statement', '').strip()
            if import_statement:
                result = add_import_statement(file_path, import_statement)
                return f"Import addition: {result}"
        
        elif action == 'insert':
            target_line = specific_fix.get('target_line')
            new_code = specific_fix.get('new_code', '').strip()
            
            if target_line and new_code:
                result = insert_text_at_line(file_path, target_line, new_code)
                return f"Line insertion: {result}"
        
        elif action == 'remove':
            target_line = specific_fix.get('target_line')
            if target_line:
                result = remove_lines_from_file(file_path, target_line)
                return f"Line removal: {result}"
        
        return "⚠️ No applicable fix action found"
        
    except Exception as e:
        return f"❌ Error applying fix: {str(e)}"
    finally:
        lock_manager.release_lock(file_path)

@tool
def fix_compilation_error_in_file(file_path: str, error_description: str, suggested_fix: str) -> str:
    """Fix a specific compilation error in a file based on error description and suggested fix."""
    lock_manager = FileLockManager(os.path.dirname(file_path))
    
    if not lock_manager.acquire_lock(file_path):
        return f"❌ Could not acquire lock for {file_path}"
    
    try:
        if not os.path.exists(file_path):
            return f"❌ File not found: {file_path}"
        
        # Read file content
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        original_content = content
        fixes_applied = []
        
        # Common compilation error patterns and their fixes
        error_fixes = {
            # Deprecated constructor usage
            "deprecated_integer_constructor": [
                ("new Integer(", "Integer.valueOf("),
                ("new Long(", "Long.valueOf("),
                ("new Double(", "Double.valueOf("),
                ("new Boolean(", "Boolean.valueOf("),
                ("new Float(", "Float.valueOf(")
            ],
            
            # Missing imports
            "missing_import": [
                # This will be handled by parsing the error description
            ],
            
            # Deprecated methods
            "deprecated_methods": [
                ("Thread.stop()", "// Thread.stop() is deprecated - use interrupt() instead"),
                (".finalize()", "// finalize() is deprecated - use try-with-resources or explicit cleanup"),
                ("System.runFinalizersOnExit", "// runFinalizersOnExit is deprecated")
            ],
            
            # JUnit 4 to 5 migration
            "junit_migration": [
                ("import org.junit.Test;", "import org.junit.jupiter.api.Test;"),
                ("import org.junit.Before;", "import org.junit.jupiter.api.BeforeEach;"),
                ("import org.junit.After;", "import org.junit.jupiter.api.AfterEach;"),
                ("import org.junit.BeforeClass;", "import org.junit.jupiter.api.BeforeAll;"),
                ("import org.junit.AfterClass;", "import org.junit.jupiter.api.AfterAll;"),
                ("import org.junit.Assert;", "import org.junit.jupiter.api.Assertions;"),
                ("@Before", "@BeforeEach"),
                ("@After", "@AfterEach"),
                ("@BeforeClass", "@BeforeAll"),
                ("@AfterClass", "@AfterAll"),
                ("Assert.assertEquals", "Assertions.assertEquals"),
                ("Assert.assertTrue", "Assertions.assertTrue"),
                ("Assert.assertFalse", "Assertions.assertFalse"),
                ("Assert.assertNull", "Assertions.assertNull"),
                ("Assert.assertNotNull", "Assertions.assertNotNull")
            ],
            
            # javax to jakarta migration
            "javax_to_jakarta": [
                ("import javax.servlet", "import jakarta.servlet"),
                ("import javax.persistence", "import jakarta.persistence"),
                ("import javax.validation", "import jakarta.validation"),
                ("import javax.annotation", "import jakarta.annotation"),
                ("import javax.inject", "import jakarta.inject"),
                ("import javax.transaction", "import jakarta.transaction")
            ]
        }
        
        # Apply suggested fix if provided
        if suggested_fix and suggested_fix != "auto":
            # Parse suggested fix format: "old_text -> new_text"
            if " -> " in suggested_fix:
                old_text, new_text = suggested_fix.split(" -> ", 1)
                old_text = old_text.strip()
                new_text = new_text.strip()
                
                if old_text in content:
                    content = content.replace(old_text, new_text)
                    fixes_applied.append(f"Applied suggested fix: {old_text} -> {new_text}")
        
        # Apply automatic fixes based on error description
        error_lower = error_description.lower()
        
        if "deprecated" in error_lower and "constructor" in error_lower:
            for old_pattern, new_pattern in error_fixes["deprecated_integer_constructor"]:
                if old_pattern in content:
                    content = content.replace(old_pattern, new_pattern)
                    fixes_applied.append(f"Fixed deprecated constructor: {old_pattern} -> {new_pattern}")
        
        if "junit" in error_lower or "test" in error_lower:
            for old_pattern, new_pattern in error_fixes["junit_migration"]:
                if old_pattern in content:
                    content = content.replace(old_pattern, new_pattern)
                    fixes_applied.append(f"JUnit migration: {old_pattern} -> {new_pattern}")
        
        if "javax" in error_lower or "jakarta" in error_lower:
            for old_pattern, new_pattern in error_fixes["javax_to_jakarta"]:
                if old_pattern in content:
                    content = content.replace(old_pattern, new_pattern)
                    fixes_applied.append(f"javax->jakarta: {old_pattern} -> {new_pattern}")
        
        if "deprecated" in error_lower and "method" in error_lower:
            for old_pattern, new_pattern in error_fixes["deprecated_methods"]:
                if old_pattern in content:
                    content = content.replace(old_pattern, new_pattern)
                    fixes_applied.append(f"Fixed deprecated method: {old_pattern} -> {new_pattern}")
        
        # Handle missing imports
        if "cannot find symbol" in error_lower or "does not exist" in error_lower:
            # Extract class name from error and suggest common imports
            import re
            class_match = re.search(r'symbol:\s*class\s+(\w+)', error_description)
            if class_match:
                class_name = class_match.group(1)
                common_imports = {
                    "Test": "import org.junit.jupiter.api.Test;",
                    "BeforeEach": "import org.junit.jupiter.api.BeforeEach;",
                    "AfterEach": "import org.junit.jupiter.api.AfterEach;",
                    "Assertions": "import org.junit.jupiter.api.Assertions;",
                    "List": "import java.util.List;",
                    "ArrayList": "import java.util.ArrayList;",
                    "HashMap": "import java.util.HashMap;",
                    "Map": "import java.util.Map;",
                    "Optional": "import java.util.Optional;",
                    "Stream": "import java.util.stream.Stream;"
                }
                
                if class_name in common_imports:
                    import_line = common_imports[class_name]
                    if import_line not in content:
                        # Add import at the top of the file after package declaration
                        lines = content.split('\n')
                        insert_pos = 0
                        for i, line in enumerate(lines):
                            if line.strip().startswith('package '):
                                insert_pos = i + 1
                                break
                        
                        lines.insert(insert_pos, import_line)
                        content = '\n'.join(lines)
                        fixes_applied.append(f"Added missing import: {import_line}")
        
        # Write the modified content back to file if changes were made
        if content != original_content:
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(content)
            
            return f"✅ Successfully applied {len(fixes_applied)} fix(es) to {file_path}:\n" + "\n".join(fixes_applied)
        else:
            return f"⚠️ No applicable fixes found for error in {file_path}: {error_description}"
        
    except Exception as e:
        return f"❌ Error fixing compilation error in {file_path}: {str(e)}"
    finally:
        lock_manager.release_lock(file_path)

@tool
def batch_update_files(project_path: str, file_pattern: str, old_text: str, new_text: str) -> str:
    """Update multiple files matching a pattern with the same find/replace operation."""
    import fnmatch
    
    try:
        updated_files = []
        error_files = []
        
        # Find all files matching the pattern
        for root, dirs, files in os.walk(project_path):
            for file in files:
                if fnmatch.fnmatch(file, file_pattern):
                    file_path = os.path.join(root, file)
                    
                    # Use the modify_file_content function
                    result = modify_file_content(file_path, old_text, new_text, all_occurrences=True)
                    
                    if result.startswith("✅"):
                        updated_files.append(file_path)
                    elif result.startswith("❌"):
                        error_files.append(f"{file_path}: {result}")
        
        result_summary = f"✅ Batch update completed:\n"
        result_summary += f"   Updated files: {len(updated_files)}\n"
        result_summary += f"   Error files: {len(error_files)}\n"
        
        if updated_files:
            result_summary += f"\nUpdated files:\n" + "\n".join([f"   - {f}" for f in updated_files[:10]])
            if len(updated_files) > 10:
                result_summary += f"\n   ... and {len(updated_files) - 10} more files"
        
        if error_files:
            result_summary += f"\nErrors:\n" + "\n".join([f"   - {e}" for e in error_files[:5]])
            if len(error_files) > 5:
                result_summary += f"\n   ... and {len(error_files) - 5} more errors"
        
        return result_summary
        
    except Exception as e:
        return f"❌ Error in batch update: {str(e)}"

@tool
def analyze_and_fix_file_errors(file_path: str, build_errors: str) -> str:
    """Analyze build errors for a specific file and apply appropriate fixes."""
    try:
        # Parse errors related to this specific file
        file_errors = []
        for line in build_errors.split('\n'):
            if file_path in line or os.path.basename(file_path) in line:
                file_errors.append(line)
        
        if not file_errors:
            return f"⚠️ No errors found for file {file_path} in the provided error log"
        
        fixes_applied = []
        
        for error_line in file_errors:
            error_lower = error_line.lower()
            
            # For unknown errors, use intelligent error fixer
            if not any(pattern in error_lower for pattern in ["cannot find symbol", "deprecated", "package does not exist"]):
                # Use intelligent error fixer for unknown errors
                intelligent_fix_result = intelligent_error_fixer(error_line)
                fixes_applied.append(f"Intelligent fix: {intelligent_fix_result}")
                continue
            
            # Identify error type and apply appropriate fix
            if "cannot find symbol" in error_lower:
                # Extract symbol name and suggest fix
                import re
                symbol_match = re.search(r'symbol:\s*(\w+)', error_line)
                if symbol_match:
                    symbol = symbol_match.group(1)
                    fix_result = fix_compilation_error_in_file(
                        file_path, 
                        error_line, 
                        f"Missing symbol: {symbol}"
                    )
                    fixes_applied.append(fix_result)
            
            elif "deprecated" in error_lower:
                fix_result = fix_compilation_error_in_file(
                    file_path, 
                    error_line, 
                    "auto"
                )
                fixes_applied.append(fix_result)
            
            elif "package does not exist" in error_lower:
                # Extract package name and fix import
                package_match = re.search(r'package ([\w.]+) does not exist', error_line)
                if package_match:
                    package = package_match.group(1)
                    # Common package migrations
                    if package.startswith('javax.'):
                        new_package = package.replace('javax.', 'jakarta.')
                        fix_result = modify_file_content(
                            file_path,
                            f"import {package}",
                            f"import {new_package}"
                        )
                        fixes_applied.append(fix_result)
        
        if fixes_applied:
            return f"✅ Applied {len(fixes_applied)} fix(es) to {file_path}:\n" + "\n".join(fixes_applied)
        else:
            return f"⚠️ No automatic fixes available for errors in {file_path}"
            
    except Exception as e:
        return f"❌ Error analyzing and fixing file {file_path}: {str(e)}"

print("✅ Enhanced file operations with intelligent error fixing implemented!")

✅ Enhanced file operations with intelligent error fixing implemented!


In [62]:
# Enhanced Agents with Large Context Memory and Intelligent Error Fixing

class AnalysisAgent:
    """LangGraph-based analysis agent with enhanced memory management."""
    
    def __init__(self, session_id: str):
        self.session_id = session_id
        self.agent_name = "analysis"
        
        # Use enhanced memory manager with 200k context
        self.memory_manager = EnhancedMemoryManager(
            session_id=session_id,
            agent_name=self.agent_name,
            max_tokens=200000,
            summarize_threshold=0.7
        )
        
        self.tools = [
            analyze_and_update_dependencies,
            check_specific_dependency,
            update_specific_dependency,
            research_migration_patterns,
            read_file_content,
            analyze_and_fix_file_errors,
            intelligent_error_fixer,  # Added intelligent error fixing
        ]
        self.agent = create_react_agent(CLAUDE_SONNET, self.tools)
    
    def analyze(self, project_path: str) -> Dict[str, Any]:
        """Analyze project for migration opportunities (interface method for orchestrator)."""
        try:
            print(f"🔍 Analyzing project: {project_path}")
            
            # Check if project exists
            if not os.path.exists(project_path):
                return {
                    "success": False,
                    "error": f"Project path not found: {project_path}",
                    "recommendations": []
                }
            
            # Check for pom.xml
            pom_path = os.path.join(project_path, "pom.xml")
            has_pom = os.path.exists(pom_path)
            
            # Check for Java files
            java_files = []
            if os.path.exists(os.path.join(project_path, "src")):
                for root, dirs, files in os.walk(os.path.join(project_path, "src")):
                    java_files.extend([f for f in files if f.endswith('.java')])
            
            # Basic dependency analysis if pom.xml exists
            dependencies_info = "No Maven project detected"
            if has_pom:
                try:
                    parser = PomParser(pom_path)
                    dependencies = parser.get_dependencies()
                    plugins = parser.get_plugins()
                    dependencies_info = f"Found {len(dependencies)} dependencies and {len(plugins)} plugins"
                except Exception as e:
                    dependencies_info = f"Could not parse pom.xml: {e}"
            
            analysis_result = {
                "success": True,
                "project_path": project_path,
                "has_maven": has_pom,
                "java_files_count": len(java_files),
                "dependencies_info": dependencies_info,
                "recommendations": [
                    "✅ Project structure detected",
                    f"✅ Found {len(java_files)} Java files",
                    "✅ Maven project detected" if has_pom else "⚠️ No pom.xml found",
                    dependencies_info,
                    "🔄 Ready for dependency analysis",
                    "🔄 Ready for deprecation scanning"
                ],
                "complexity": "moderate" if len(java_files) > 10 else "simple",
                "estimated_duration": "15-30 minutes"
            }
            
            print("✅ Project analysis completed successfully")
            return analysis_result
            
        except Exception as e:
            print(f"❌ Analysis failed: {e}")
            return {
                "success": False,
                "error": str(e),
                "recommendations": ["❌ Analysis failed - check project path and permissions"]
            }
    
    def analyze_project(self, state: AgentState) -> AgentState:
        """Analyze project with enhanced dependency checking and large context memory."""
        project_path = state["project_path"]
        
        # Get conversation history for context
        conversation_buffer = self.memory_manager.get_conversation_buffer()
        
        analysis_prompt = f"""
        Analyze the Java project at {project_path} for migration from Java 11 to 21.
        
        Previous context: {conversation_buffer}
        
        Please:
        1. Read key project files (pom.xml, main Java files) using read_file_content
        2. Use analyze_and_update_dependencies to check and update dependencies dynamically
        3. Research migration patterns for any identified technologies
        4. If you encounter any unknown errors during analysis, use intelligent_error_fixer
        5. Assess overall migration complexity
        6. Provide specific recommendations
        
        For any compilation or dependency issues you encounter:
        - First try analyze_and_fix_file_errors for systematic fixes
        - For unknown/complex errors, use intelligent_error_fixer with the full error message
        - This will automatically parse the error, read context, and apply LLM-suggested fixes
        
        Use the tools available to gather comprehensive information.
        
        Current memory stats: {self.memory_manager.get_memory_stats()}
        """
        
        try:
            result = self.agent.invoke({
                "messages": [{"role": "user", "content": analysis_prompt}]
            })
            
            result_content = result['messages'][-1]['content']
            
            # Add to enhanced memory
            self.memory_manager.add_message(analysis_prompt, "human")
            self.memory_manager.add_message(result_content, "assistant")
            
            state["messages"].append({
                "role": "assistant", 
                "content": f"Analysis completed: {result_content}"
            })
            state["last_action_result"] = "analysis_success"
            state["error_count"] = 0
            
            # Log memory usage
            memory_stats = self.memory_manager.get_memory_stats()
            print(f"📊 Analysis Agent Memory: {memory_stats['estimated_tokens']:,}/{memory_stats['max_tokens']:,} tokens ({memory_stats['usage_percentage']:.1f}%)")
            
        except Exception as e:
            error_msg = f"Analysis failed: {str(e)}"
            self.memory_manager.add_message(f"Analysis error: {error_msg}", "assistant")
            
            state["messages"].append({
                "role": "assistant",
                "content": error_msg
            })
            state["last_action_result"] = "analysis_failed"
            state["error_count"] += 1
        
        return state

class CodeMigrationAgent:
    """LangGraph-based code migration agent with enhanced memory management and intelligent error fixing."""
    
    def __init__(self, session_id: str):
        self.session_id = session_id
        self.agent_name = "code_migration"
        
        # Use enhanced memory manager with 200k context
        self.memory_manager = EnhancedMemoryManager(
            session_id=session_id,
            agent_name=self.agent_name,
            max_tokens=200000,
            summarize_threshold=0.65  # Slightly lower threshold for migration agent
        )
        
        self.tools = [
            run_maven_tests,
            run_maven_build,
            run_test_fix_cycle,
            run_build_fix_cycle,
            analyze_and_update_dependencies,
            auto_fix_common_test_issues,
            auto_fix_build_errors,
            # Enhanced file operation tools with intelligent error fixing
            read_file_content,
            write_file_content,
            modify_file_content,
            add_import_statement,
            fix_compilation_error_in_file,
            batch_update_files,
            analyze_and_fix_file_errors,
            intelligent_error_fixer,  # Key addition for unknown errors
        ]
        self.agent = create_react_agent(CLAUDE_SONNET, self.tools)
    
    def migrate_dependencies(self, state: AgentState) -> AgentState:
        """Migrate dependencies with automated testing, fixing, and intelligent error resolution."""
        project_path = state["project_path"]
        
        # Get rich context from memory
        conversation_buffer = self.memory_manager.get_conversation_buffer()
        recent_messages = self.memory_manager.get_recent_messages(10)
        
        migration_prompt = f"""
        Migrate the dependencies for the Java project at {project_path}.
        
        Previous context and learnings:
        {conversation_buffer}
        
        Recent activity:
        {[msg.content[:200] for msg in recent_messages]}
        
        Please execute this enhanced migration workflow:
        
        1. **Dependency Analysis & Updates**: Use analyze_and_update_dependencies to get real Maven Central versions
        2. **File Analysis**: Read project files to understand current state using read_file_content
        3. **Systematic Fixes**: Use file modification tools to directly fix compilation errors
        4. **Intelligent Error Resolution**: For ANY unknown/complex errors encountered:
           - Use intelligent_error_fixer(error_message) - it will automatically:
             * Parse the error to find file/line
             * Read 40-50 lines of context around the error
             * Send to LLM for analysis and get specific fix instructions
             * Apply the fix directly to the file
        5. **Testing & Validation**: Run tests after updates and fix failures using file operations
        6. **Build Validation**: Run builds and fix errors using intelligent error fixing
        7. **Retry Cycles**: Use automated retry cycles for systematic resolution
        
        **CRITICAL**: Whenever you encounter an error you don't recognize or can't categorize:
        - Copy the FULL error message
        - Call intelligent_error_fixer(full_error_message)
        - It will handle file parsing, context reading, LLM analysis, and automatic fixing
        
        Focus on these key migrations with intelligent error resolution:
        - javax to jakarta namespace (fix any resulting errors intelligently)
        - JUnit 4 to JUnit 5 (resolve complex test migration issues)
        - Spring Boot 2.x to 3.x compatibility (handle configuration changes)
        - Maven dependency updates (resolve version conflicts intelligently)
        
        Current memory usage: {self.memory_manager.get_memory_stats()['usage_percentage']:.1f}%
        """
        
        try:
            result = self.agent.invoke({
                "messages": [{"role": "user", "content": migration_prompt}]
            })
            
            result_content = result['messages'][-1]['content']
            
            # Add to enhanced memory with detailed context
            self.memory_manager.add_message(migration_prompt, "human")
            self.memory_manager.add_message(result_content, "assistant")
            
            state["messages"].append({
                "role": "assistant",
                "content": f"Dependencies migrated with intelligent error fixing: {result_content}"
            })
            state["last_action_result"] = "migration_success"
            state["error_count"] = 0
            
            # Check if summarization occurred
            memory_stats = self.memory_manager.get_memory_stats()
            print(f"📊 Migration Agent Memory: {memory_stats['estimated_tokens']:,}/{memory_stats['max_tokens']:,} tokens ({memory_stats['usage_percentage']:.1f}%)")
            if memory_stats['summarization_count'] > 0:
                print(f"🔄 Summarizations performed: {memory_stats['summarization_count']}")
            
        except Exception as e:
            error_msg = f"Migration failed: {str(e)}"
            self.memory_manager.add_message(f"Migration error: {error_msg}", "assistant")
            
            state["messages"].append({
                "role": "assistant",
                "content": error_msg
            })
            state["last_action_result"] = "migration_failed"
            state["error_count"] += 1
        
        return state
    
    def apply_openrewrite_recipes(self, state: AgentState) -> AgentState:
        """Apply OpenRewrite recipes with comprehensive context and intelligent error fixing."""
        project_path = state["project_path"]
        
        # Rich context from memory
        conversation_buffer = self.memory_manager.get_conversation_buffer()
        
        recipe_prompt = f"""
        Apply OpenRewrite recipes to migrate the Java code at {project_path}.
        
        Full context and previous learnings:
        {conversation_buffer}
        
        Apply these recipes in order, with intelligent error resolution:
        1. Java 11 to 17 migration
        2. Java 17 to 21 migration  
        3. javax to jakarta migration
        4. JUnit 4 to 5 migration
        5. Spring Boot 3.x upgrade
        
        **Enhanced Error Resolution Workflow**:
        After each recipe and at every error encountered:
        - Run tests and builds to identify issues
        - For known error patterns: use analyze_and_fix_file_errors
        - For unknown/complex errors: use intelligent_error_fixer(full_error_message)
        - The intelligent_error_fixer will:
          * Parse any Maven/compiler error to extract file:line
          * Read context around the problematic code  
          * Send comprehensive context to LLM for analysis
          * Apply the suggested fix automatically
        - Use batch_update_files for systematic replacements
        - Use the automated fix cycles for systematic resolution
        - Document what worked and what didn't in memory
        
        **Key Integration Points**:
        - After each OpenRewrite recipe, immediately check for compilation errors
        - If errors occur, don't guess - use intelligent_error_fixer
        - For test failures, use intelligent error fixing if standard patterns don't work
        - For build issues, intelligent error fixing can handle project-specific problems
        
        You have full file system access and intelligent error resolution.
        Use these capabilities aggressively to ensure migration success.
        
        If any recipe fails after 3 attempts with intelligent fixing, document clearly for escalation.
        
        Memory status: {self.memory_manager.get_memory_stats()['usage_percentage']:.1f}% used
        """
        
        try:
            result = self.agent.invoke({
                "messages": [{"role": "user", "content": recipe_prompt}]
            })
            
            result_content = result['messages'][-1]['content']
            
            # Store comprehensive results
            self.memory_manager.add_message(recipe_prompt, "human")
            self.memory_manager.add_message(result_content, "assistant")
            
            state["messages"].append({
                "role": "assistant",
                "content": f"OpenRewrite recipes applied with intelligent error fixing: {result_content}"
            })
            state["last_action_result"] = "recipes_success"
            state["error_count"] = 0
            
        except Exception as e:
            error_msg = f"Recipe application failed: {str(e)}"
            self.memory_manager.add_message(f"Recipe error: {error_msg}", "assistant")
            
            state["messages"].append({
                "role": "assistant",
                "content": error_msg
            })
            state["last_action_result"] = "recipes_failed"
            state["error_count"] += 1
        
        return state

class ValidationAgent:
    """LangGraph-based validation agent with enhanced memory management and intelligent error fixing."""
    
    def __init__(self, session_id: str):
        self.session_id = session_id
        self.agent_name = "validation"
        
        # Use enhanced memory manager with 200k context
        self.memory_manager = EnhancedMemoryManager(
            session_id=session_id,
            agent_name=self.agent_name,
            max_tokens=200000,
            summarize_threshold=0.75  # Higher threshold for validation agent
        )
        
        self.tools = [
            run_maven_tests,
            run_maven_build,
            analyze_test_failures_with_ai,
            analyze_build_errors_with_ai,
            search_build_error_solutions,
            # Enhanced file operation tools with intelligent error fixing
            read_file_content,
            modify_file_content,
            fix_compilation_error_in_file,
            analyze_and_fix_file_errors,
            batch_update_files,
            intelligent_error_fixer,  # Critical for unknown validation issues
        ]
        self.agent = create_react_agent(CLAUDE_SONNET, self.tools)
    
    def validate_migration(self, state: AgentState) -> AgentState:
        """Validate the migration with comprehensive testing, full context, and intelligent error fixing."""
        project_path = state["project_path"]
        
        # Get complete migration history
        conversation_buffer = self.memory_manager.get_conversation_buffer()
        
        validation_prompt = f"""
        Validate the completed migration for the Java project at {project_path}.
        
        Complete migration history and context:
        {conversation_buffer}
        
        Please perform comprehensive validation with intelligent error resolution:
        
        **Validation Workflow with Intelligent Fixing**:
        1. **Test Execution**: Run comprehensive tests and analyze failures with AI
        2. **Build Validation**: Run full build and check for errors
        3. **Error Analysis**: Use AI analysis for initial issue identification
        4. **Intelligent Error Resolution**: For ANY test or build failures:
           - Use intelligent_error_fixer(full_error_message) for unknown issues
           - This automatically handles:
             * Parsing test failure messages to find problematic files/lines
             * Reading code context around failing assertions or compilation errors
             * Getting LLM analysis of what's wrong and how to fix it
             * Applying fixes directly to test or source files
        5. **Web Research**: Search for solutions to complex problems using web research
        6. **Systematic Fixes**: Use file operation tools for systematic corrections
        7. **Re-validation**: Re-run tests and builds after fixes to ensure success
        8. **Final Report**: Provide comprehensive migration report
        
        **Enhanced Validation Criteria with Auto-Fixing**:
        - All tests must pass (use intelligent_error_fixer for unknown test failures)
        - Build must be successful (use intelligent error fixing for complex build errors)
        - No deprecated API warnings (fix using file modifications + intelligent fixing)
        - Java 21 features properly utilized
        - All previous issues resolved using intelligent analysis
        
        **Critical Integration Points**:
        - Test failures with unclear error messages → intelligent_error_fixer
        - Compilation errors in migrated code → intelligent_error_fixer  
        - Configuration issues after Spring Boot upgrade → intelligent_error_fixer
        - JUnit 5 migration issues → intelligent_error_fixer
        - Any "cannot resolve" or complex dependency issues → intelligent_error_fixer
        
        You have full access to read/modify project files AND intelligent error resolution.
        Use these capabilities to ensure the migration is completely successful.
        
        Memory: {self.memory_manager.get_memory_stats()['usage_percentage']:.1f}% used, {self.memory_manager.get_memory_stats()['summarization_count']} summarizations
        """
        
        try:
            result = self.agent.invoke({
                "messages": [{"role": "user", "content": validation_prompt}]
            })
            
            result_content = result['messages'][-1]['content']
            
            # Store final validation results
            self.memory_manager.add_message(validation_prompt, "human")
            self.memory_manager.add_message(result_content, "assistant")
            
            state["messages"].append({
                "role": "assistant",
                "content": f"Validation completed with intelligent error fixing: {result_content}"
            })
            state["last_action_result"] = "validation_success"
            state["error_count"] = 0
            
            # Final memory stats
            memory_stats = self.memory_manager.get_memory_stats()
            print(f"📊 Final Validation Memory: {memory_stats['estimated_tokens']:,}/{memory_stats['max_tokens']:,} tokens")
            print(f"🔄 Total summarizations: {memory_stats['summarization_count']}")
            
        except Exception as e:
            error_msg = f"Validation failed: {str(e)}"
            self.memory_manager.add_message(f"Validation error: {error_msg}", "assistant")
            
            state["messages"].append({
                "role": "assistant",
                "content": error_msg
            })
            state["last_action_result"] = "validation_failed"
            state["error_count"] += 1
        
        return state

class DeprecationAgent:
    """Specialized agent for detecting and replacing deprecated APIs, plugins, and patterns with intelligent error fixing."""
    
    def __init__(self, session_id: str):
        self.session_id = session_id
        self.agent_name = "deprecation_detection"
        
        # Use enhanced memory manager with large context
        self.memory_manager = EnhancedMemoryManager(
            session_id=session_id,
            agent_name=self.agent_name,
            max_tokens=200000,
            summarize_threshold=0.7
        )
        
        self.tools = [
            run_maven_with_deprecation_analysis,
            analyze_deprecated_apis_with_llm,
            scan_maven_plugins_deprecation,
            auto_replace_simple_deprecations,
            generate_deprecation_report,
            research_migration_patterns,  # For complex deprecations
            # Enhanced file operation tools with intelligent error fixing
            read_file_content,
            modify_file_content,
            fix_compilation_error_in_file,
            batch_update_files,
            analyze_and_fix_file_errors,
            intelligent_error_fixer,  # Critical for complex deprecation issues
        ]
        self.agent = create_react_agent(CLAUDE_SONNET, self.tools)
        
        # Track deprecation items found
        self.deprecation_items = []
        self.auto_fixes_applied = []
        self.manual_review_needed = []
    
    def detect_and_analyze_deprecations(self, state: AgentState) -> AgentState:
        """Comprehensive deprecation detection and analysis with intelligent error fixing for complex cases."""
        project_path = state["project_path"]
        
        # Get context from memory
        conversation_buffer = self.memory_manager.get_conversation_buffer()
        
        deprecation_prompt = f"""
        Perform comprehensive deprecation analysis for the Java project at {project_path}.
        
        Previous migration context:
        {conversation_buffer}
        
        Please execute this enhanced deprecation detection workflow:
        
        **Deprecation Analysis with Intelligent Error Resolution**:
        1. **Maven Build Analysis**: Run Maven with deprecation warnings enabled
        2. **LLM Analysis**: Analyze found deprecations and suggest modern replacements
        3. **Plugin Scanning**: Check for deprecated Maven plugins using real Maven Central data
        4. **Direct File Modifications**: Use file operation tools to apply fixes directly
        5. **Intelligent Error Resolution**: For complex deprecation issues:
           - When automated replacements fail → use intelligent_error_fixer
           - When deprecation warnings point to unclear code → intelligent_error_fixer
           - When replacement APIs have different signatures → intelligent_error_fixer
           - When Spring/framework upgrades break existing patterns → intelligent_error_fixer
        6. **Report Generation**: Create detailed deprecation report
        
        **Enhanced Deprecation Handling**:
        For each deprecated item found:
        - Assess migration complexity (simple/moderate/complex)
        - Determine if it's a breaking change
        - **Apply fixes directly** using file operation tools
        - **For complex cases**: Use intelligent_error_fixer(error_message) when:
          * Automatic replacements cause compilation errors
          * Deprecated APIs have changed signatures significantly
          * Framework upgrade patterns are unclear
          * Multiple interdependent deprecations cause cascading issues
        - Provide specific replacement recommendations
        - Include code examples and prioritize by importance
        
        **Intelligent Fixing Integration Points**:
        - After applying javax→jakarta replacements: check for compilation errors, fix intelligently
        - After JUnit 4→5 migrations: resolve complex annotation/assertion issues intelligently  
        - After Spring Boot upgrades: fix configuration/autowiring issues intelligently
        - After Maven plugin updates: resolve build configuration issues intelligently
        
        Focus areas with intelligent error support:
        - Java 8-21 deprecated APIs (fix compilation issues intelligently)
        - Spring Framework deprecated patterns (resolve complex config issues)
        - JUnit 4 → 5 migration opportunities (fix complex test migration issues)
        - Maven plugin updates (resolve build issues intelligently)
        - Legacy annotation patterns (fix framework integration issues)
        
        **KEY**: Don't just identify deprecations - FIX THEM and resolve any resulting issues intelligently!
        
        Memory usage: {self.memory_manager.get_memory_stats()['usage_percentage']:.1f}%
        """
        
        try:
            result = self.agent.invoke({
                "messages": [{"role": "user", "content": deprecation_prompt}]
            })
            
            result_content = result['messages'][-1]['content']
            
            # Store in enhanced memory
            self.memory_manager.add_message(deprecation_prompt, "human")
            self.memory_manager.add_message(result_content, "assistant")
            
            # Extract deprecation items for later reference
            self._extract_deprecation_items(result_content)
            
            state["messages"].append({
                "role": "assistant",
                "content": f"Deprecation analysis completed with intelligent error fixing: {result_content}"
            })
            state["last_action_result"] = "deprecation_analysis_success"
            state["error_count"] = 0
            
            # Log memory and findings
            memory_stats = self.memory_manager.get_memory_stats()
            print(f"📊 Deprecation Agent Memory: {memory_stats['estimated_tokens']:,}/{memory_stats['max_tokens']:,} tokens ({memory_stats['usage_percentage']:.1f}%)")
            print(f"🔍 Deprecation items found: {len(self.deprecation_items)}")
            print(f"🔧 Auto-fixes applied: {len(self.auto_fixes_applied)}")
            print(f"👁️ Items needing manual review: {len(self.manual_review_needed)}")
            
        except Exception as e:
            error_msg = f"Deprecation analysis failed: {str(e)}"
            self.memory_manager.add_message(f"Deprecation analysis error: {error_msg}", "assistant")
            
            state["messages"].append({
                "role": "assistant",
                "content": error_msg
            })
            state["last_action_result"] = "deprecation_analysis_failed"
            state["error_count"] += 1
        
        return state
    
    def _extract_deprecation_items(self, result_content: str):
        """Extract and categorize deprecation items from agent results."""
        # This is a simplified extraction - in practice you'd parse the JSON response
        lines = result_content.split('\n')
        
        for line in lines:
            if 'deprecated' in line.lower():
                if 'simple' in line.lower() or 'automatic' in line.lower():
                    self.auto_fixes_applied.append(line.strip())
                elif 'complex' in line.lower() or 'manual' in line.lower():
                    self.manual_review_needed.append(line.strip())
                else:
                    self.deprecation_items.append(line.strip())
    
    def get_deprecation_summary(self) -> Dict[str, Any]:
        """Get summary of deprecation analysis results."""
        return {
            "total_deprecation_items": len(self.deprecation_items),
            "auto_fixes_applied": len(self.auto_fixes_applied),
            "manual_review_needed": len(self.manual_review_needed),
            "memory_usage": self.memory_manager.get_memory_stats(),
            "deprecation_categories": {
                "api_deprecations": [item for item in self.deprecation_items if 'api' in item.lower()],
                "plugin_deprecations": [item for item in self.deprecation_items if 'plugin' in item.lower()],
                "annotation_deprecations": [item for item in self.deprecation_items if 'annotation' in item.lower()]
            }
        }

print("✅ Enhanced agents with intelligent error fixing integration implemented successfully!")

✅ Enhanced agents with intelligent error fixing integration implemented successfully!


In [63]:
# Enhanced Deprecation Detection Agent

class EnhancedDeprecationAgent:
    """Specialized agent for detecting and replacing deprecated APIs, plugins, and patterns."""
    
    def __init__(self, session_id: str):
        self.session_id = session_id
        self.agent_name = "deprecation_detection"
        
        # Use enhanced memory manager with large context
        self.memory_manager = EnhancedMemoryManager(
            session_id=session_id,
            agent_name=self.agent_name,
            max_tokens=200000,
            summarize_threshold=0.7
        )
        
        self.tools = [
            run_maven_with_deprecation_analysis,
            analyze_deprecated_apis_with_llm,
            scan_maven_plugins_deprecation,
            auto_replace_simple_deprecations,
            generate_deprecation_report,
            research_migration_patterns  # For complex deprecations
        ]
        self.agent = create_react_agent(CLAUDE_SONNET, self.tools)
        
        # Track deprecation items found
        self.deprecation_items = []
        self.auto_fixes_applied = []
        self.manual_review_needed = []
    
    def detect_and_analyze_deprecations(self, state: AgentState) -> AgentState:
        """Comprehensive deprecation detection and analysis."""
        project_path = state["project_path"]
        
        # Get context from memory
        conversation_buffer = self.memory_manager.get_conversation_buffer()
        
        deprecation_prompt = f"""
        Perform comprehensive deprecation analysis for the Java project at {project_path}.
        
        Previous migration context:
        {conversation_buffer}
        
        Please execute this deprecation detection workflow:
        
        1. **Maven Build Analysis**: Run Maven with deprecation warnings enabled
        2. **LLM Analysis**: Analyze found deprecations and suggest modern replacements
        3. **Plugin Scanning**: Check for deprecated Maven plugins
        4. **Automatic Fixes**: Apply safe automatic replacements
        5. **Report Generation**: Create detailed deprecation report
        
        For each deprecated item found:
        - Assess the complexity of migration (simple/moderate/complex)
        - Determine if it's a breaking change
        - Provide specific replacement recommendations
        - Include code examples where helpful
        - Prioritize by importance and risk
        
        Focus on:
        - Java 8-21 deprecated APIs
        - Spring Framework deprecated patterns  
        - JUnit 4 → 5 migration opportunities
        - Maven plugin updates
        - Legacy annotation patterns
        - Deprecated HTTP clients, date/time APIs, etc.
        
        Use all available tools to provide comprehensive analysis.
        
        Memory usage: {self.memory_manager.get_memory_stats()['usage_percentage']:.1f}%
        """
        
        try:
            result = self.agent.invoke({
                "messages": [{"role": "user", "content": deprecation_prompt}]
            })
            
            result_content = result['messages'][-1]['content']
            
            # Store in enhanced memory
            self.memory_manager.add_message(deprecation_prompt, "human")
            self.memory_manager.add_message(result_content, "assistant")
            
            # Extract deprecation items for later reference
            self._extract_deprecation_items(result_content)
            
            state["messages"].append({
                "role": "assistant",
                "content": f"Deprecation analysis completed: {result_content}"
            })
            state["last_action_result"] = "deprecation_analysis_success"
            state["error_count"] = 0
            
            # Log memory and findings
            memory_stats = self.memory_manager.get_memory_stats()
            print(f"📊 Deprecation Agent Memory: {memory_stats['estimated_tokens']:,}/{memory_stats['max_tokens']:,} tokens ({memory_stats['usage_percentage']:.1f}%)")
            print(f"🔍 Deprecation items found: {len(self.deprecation_items)}")
            print(f"🔧 Auto-fixes applied: {len(self.auto_fixes_applied)}")
            print(f"👁️ Items needing manual review: {len(self.manual_review_needed)}")
            
        except Exception as e:
            error_msg = f"Deprecation analysis failed: {str(e)}"
            self.memory_manager.add_message(f"Deprecation analysis error: {error_msg}", "assistant")
            
            state["messages"].append({
                "role": "assistant",
                "content": error_msg
            })
            state["last_action_result"] = "deprecation_analysis_failed"
            state["error_count"] += 1
        
        return state
    
    def _extract_deprecation_items(self, result_content: str):
        """Extract and categorize deprecation items from agent results."""
        # This is a simplified extraction - in practice you'd parse the JSON response
        lines = result_content.split('\n')
        
        for line in lines:
            if 'deprecated' in line.lower():
                if 'simple' in line.lower() or 'automatic' in line.lower():
                    self.auto_fixes_applied.append(line.strip())
                elif 'complex' in line.lower() or 'manual' in line.lower():
                    self.manual_review_needed.append(line.strip())
                else:
                    self.deprecation_items.append(line.strip())
    
    def get_deprecation_summary(self) -> Dict[str, Any]:
        """Get summary of deprecation analysis results."""
        return {
            "total_deprecation_items": len(self.deprecation_items),
            "auto_fixes_applied": len(self.auto_fixes_applied),
            "manual_review_needed": len(self.manual_review_needed),
            "memory_usage": self.memory_manager.get_memory_stats(),
            "deprecation_categories": {
                "api_deprecations": [item for item in self.deprecation_items if 'api' in item.lower()],
                "plugin_deprecations": [item for item in self.deprecation_items if 'plugin' in item.lower()],
                "annotation_deprecations": [item for item in self.deprecation_items if 'annotation' in item.lower()]
            }
        }

print("✅ Enhanced Deprecation Detection Agent implemented successfully!")

✅ Enhanced Deprecation Detection Agent implemented successfully!


In [64]:
# Enhanced Memory Management with Large Context Windows

class EnhancedMemoryManager:
    """Advanced memory manager with large context windows and proactive summarization."""
    
    def __init__(self, session_id: str, agent_name: str, 
                 max_tokens: int = 200000, 
                 summarize_threshold: float = 0.7):
        self.session_id = session_id
        self.agent_name = agent_name
        self.max_tokens = max_tokens
        self.summarize_threshold = summarize_threshold
        self.summarize_trigger = int(max_tokens * summarize_threshold)  # 140k at 70%
        
        self.redis_client = redis.from_url(REDIS_URL)
        self.llm = CLAUDE_SONNET
        self.token_counter = 0
        self.summarization_count = 0
        
        # Setup Redis-backed chat history
        self.message_history = RedisChatMessageHistory(
            url=REDIS_URL,
            ttl=86400,
            session_id=f"{self.session_id}_{self.agent_name}"
        )
        
        # Enhanced memory with large context
        self.memory = ConversationSummaryBufferMemory(
            chat_memory=self.message_history,
            max_token_limit=self.max_tokens,
            return_messages=True,
            llm=self.llm
        )
        
        print(f"📊 Enhanced Memory initialized for {agent_name}:")
        print(f"   • Max tokens: {self.max_tokens:,}")
        print(f"   • Summarize at: {self.summarize_trigger:,} tokens ({summarize_threshold*100:.0f}%)")
    
    def add_message(self, message: str, role: str = "assistant"):
        """Add message with proactive summarization."""
        # Estimate tokens (rough approximation: 1 token ≈ 4 characters)
        estimated_tokens = len(message) // 4
        self.token_counter += estimated_tokens
        
        # Add to memory
        if role == "human":
            self.memory.chat_memory.add_user_message(message)
        else:
            self.memory.chat_memory.add_ai_message(message)
        
        # Check if we need proactive summarization
        if self.token_counter >= self.summarize_trigger:
            self._proactive_summarize()
        
        # Log memory stats
        self._log_memory_stats()
    
    def _proactive_summarize(self):
        """Proactively summarize when approaching token limit."""
        print(f"🔄 Proactive summarization triggered at {self.token_counter:,} tokens")
        
        try:
            # Get current messages
            messages = self.memory.chat_memory.messages
            
            if len(messages) > 10:  # Only summarize if we have enough content
                # Keep recent messages (last 5) and summarize the rest
                recent_messages = messages[-5:]
                messages_to_summarize = messages[:-5]
                
                # Create comprehensive summary
                summary_prompt = f"""
                Summarize the following conversation history for a Java migration agent session.
                
                Session: {self.session_id}
                Agent: {self.agent_name}
                Summarization #{self.summarization_count + 1}
                
                Focus on:
                1. Key actions taken and their results
                2. Important findings and decisions
                3. Errors encountered and how they were resolved
                4. Current state and progress
                5. Any patterns or learnings discovered
                
                Messages to summarize:
                {self._format_messages_for_summary(messages_to_summarize)}
                
                Provide a comprehensive but concise summary that preserves all important context.
                """
                
                summary_response = self.llm.invoke(summary_prompt)
                summary = summary_response.content
                
                # Clear old messages and add summary
                self.message_history.clear()
                self.message_history.add_ai_message(f"[SUMMARY #{self.summarization_count + 1}]: {summary}")
                
                # Re-add recent messages
                for msg in recent_messages:
                    if hasattr(msg, 'type'):
                        if msg.type == 'human':
                            self.message_history.add_user_message(msg.content)
                        else:
                            self.message_history.add_ai_message(msg.content)
                
                # Reset token counter and increment summarization count
                self.token_counter = len(summary) // 4 + sum(len(msg.content) // 4 for msg in recent_messages)
                self.summarization_count += 1
                
                print(f"✅ Summarization complete. Token count reduced to ~{self.token_counter:,}")
                print(f"📝 Summary length: {len(summary):,} characters")
                
                # Log summarization event
                self._log_summarization_event(len(messages_to_summarize), len(summary))
                
        except Exception as e:
            print(f"❌ Summarization failed: {e}")
    
    def _format_messages_for_summary(self, messages) -> str:
        """Format messages for summarization prompt."""
        formatted = []
        for i, msg in enumerate(messages):
            role = "Human" if hasattr(msg, 'type') and msg.type == 'human' else "Assistant"
            content = msg.content[:1000]  # Limit length
            formatted.append(f"{i+1}. {role}: {content}")
        return "\n".join(formatted)
    
    def _log_memory_stats(self):
        """Log current memory statistics."""
        memory_key = f"memory_stats:{self.session_id}:{self.agent_name}"
        stats = {
            "timestamp": datetime.now().isoformat(),
            "estimated_tokens": self.token_counter,
            "max_tokens": self.max_tokens,
            "usage_percentage": (self.token_counter / self.max_tokens) * 100,
            "summarization_count": self.summarization_count,
            "messages_count": len(self.memory.chat_memory.messages)
        }
        self.redis_client.set(memory_key, json.dumps(stats), ex=86400)
    
    def _log_summarization_event(self, messages_summarized: int, summary_length: int):
        """Log summarization events for analysis."""
        event_key = f"summarization_log:{self.session_id}:{self.agent_name}"
        event = {
            "timestamp": datetime.now().isoformat(),
            "summarization_number": self.summarization_count,
            "messages_summarized": messages_summarized,
            "summary_length": summary_length,
            "tokens_before": self.token_counter + (messages_summarized * 100),  # Rough estimate
            "tokens_after": self.token_counter
        }
        self.redis_client.lpush(event_key, json.dumps(event))
        self.redis_client.expire(event_key, 86400)
    
    def get_memory_stats(self) -> Dict[str, Any]:
        """Get current memory statistics."""
        return {
            "session_id": self.session_id,
            "agent_name": self.agent_name,
            "estimated_tokens": self.token_counter,
            "max_tokens": self.max_tokens,
            "usage_percentage": (self.token_counter / self.max_tokens) * 100,
            "summarize_threshold": self.summarize_trigger,
            "summarization_count": self.summarization_count,
            "messages_count": len(self.memory.chat_memory.messages),
            "ready_for_summarization": self.token_counter >= self.summarize_trigger
        }
    
    def force_summarize(self):
        """Manually trigger summarization."""
        print(f"🔧 Manual summarization triggered")
        self._proactive_summarize()
    
    def get_conversation_buffer(self):
        """Get the current conversation buffer."""
        return self.memory.buffer
    
    def get_recent_messages(self, count: int = 5):
        """Get the most recent messages."""
        messages = self.memory.chat_memory.messages
        return messages[-count:] if len(messages) >= count else messages

print("✅ Enhanced Memory Manager with large context windows implemented!")

✅ Enhanced Memory Manager with large context windows implemented!


In [65]:
# Updated Migration Orchestrator with Clean Agent References

class MigrationOrchestrator:
    """Enhanced orchestrator with configurable phases and clean agent references."""
    
    def __init__(self, project_path: str, session_id: str = None, config: MigrationConfiguration = None):
        self.project_path = project_path
        self.session_id = session_id or f"migration_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        self.redis_client = redis.from_url(REDIS_URL)
        self.lock_manager = FileLockManager(project_path)
        
        # Use provided configuration or create default
        self.config = config or MigrationConfiguration()
        
        # Initialize all agents with clean names
        self.analysis_agent = AnalysisAgent(self.session_id)
        self.deprecation_agent = DeprecationAgent(self.session_id)
        self.migration_agent = CodeMigrationAgent(self.session_id)
        self.validation_agent = ValidationAgent(self.session_id)
        
        # Create configurable workflow
        self.workflow = self._create_configurable_workflow()
        
        print(f"🎛️ Migration Orchestrator initialized")
        enabled_phases = self.config.get_enabled_phases()
        print(f"📋 Enabled phases: {[phase.value for phase in enabled_phases]}")
        
    def _create_configurable_workflow(self) -> StateGraph:
        """Create workflow based on configuration."""
        workflow = StateGraph(AgentState)
        enabled_phases = self.config.get_enabled_phases()
        
        # Dynamically add enabled phases
        phase_methods = {
            MigrationPhaseType.ANALYSIS: self._analysis_phase,
            MigrationPhaseType.DEPRECATION_DETECTION: self._deprecation_detection_phase,
            MigrationPhaseType.DEPENDENCY_UPDATE: self._dependency_update_phase,
            MigrationPhaseType.CODE_MIGRATION: self._code_migration_phase,
            MigrationPhaseType.TESTING_VALIDATION: self._testing_validation_phase,
            MigrationPhaseType.PERFORMANCE_VALIDATION: self._performance_validation_phase,
            MigrationPhaseType.FINAL_CLEANUP: self._final_cleanup_phase
        }
        
        # Add nodes for enabled phases
        for phase in enabled_phases:
            if phase in phase_methods:
                workflow.add_node(phase.value, phase_methods[phase])
        
        # Add common nodes
        workflow.add_node("escalate", self._escalate_phase)
        workflow.add_node("complete", self._complete_phase)
        
        # Set entry point to first enabled phase
        if enabled_phases:
            workflow.set_entry_point(enabled_phases[0].value)
        
        # Create conditional edges between enabled phases
        for i, current_phase in enumerate(enabled_phases):
            next_phase = enabled_phases[i + 1] if i + 1 < len(enabled_phases) else None
            
            if next_phase:
                workflow.add_conditional_edges(
                    current_phase.value,
                    self._should_continue,
                    {
                        "continue": next_phase.value,
                        "escalate": "escalate",
                        "complete": "complete"
                    }
                )
            else:
                # Last phase goes to validation or completion
                workflow.add_conditional_edges(
                    current_phase.value,
                    self._should_continue,
                    {
                        "continue": "complete",
                        "escalate": "escalate", 
                        "complete": "complete"
                    }
                )
        
        workflow.add_edge("escalate", "complete")
        workflow.add_edge("complete", END)
        
        return workflow.compile()
    
    def _analysis_phase(self, state: AgentState) -> AgentState:
        """Execute enhanced analysis phase."""
        print("🔍 Analysis Phase")
        state["current_phase"] = "analysis"
        self._create_git_checkpoint("Pre-analysis checkpoint")
        state = self.analysis_agent.analyze_project(state)
        return state
    
    def _deprecation_detection_phase(self, state: AgentState) -> AgentState:
        """Execute deprecation detection and modernization phase."""
        print("🔍 Deprecation Detection & Modernization Phase")
        state["current_phase"] = "deprecation_detection"
        
        # Create checkpoint before deprecation analysis
        self._create_git_checkpoint("Pre-deprecation-analysis checkpoint")
        
        # Run comprehensive deprecation analysis
        state = self.deprecation_agent.detect_and_analyze_deprecations(state)
        
        # Log deprecation summary
        summary = self.deprecation_agent.get_deprecation_summary()
        print(f"📊 Deprecation Summary:")
        print(f"   • Total items found: {summary['total_deprecation_items']}")
        print(f"   • Auto-fixes applied: {summary['auto_fixes_applied']}")
        print(f"   • Manual review needed: {summary['manual_review_needed']}")
        
        return state
    
    def _dependency_update_phase(self, state: AgentState) -> AgentState:
        """Execute dependency update phase."""
        print("📦 Dependency Update Phase")
        state["current_phase"] = "dependency_update"
        self._create_git_checkpoint("Pre-dependency-update checkpoint")
        state = self.migration_agent.migrate_dependencies(state)
        return state
    
    def _code_migration_phase(self, state: AgentState) -> AgentState:
        """Execute code migration phase."""
        print("🔧 Code Migration Phase")
        state["current_phase"] = "code_migration"
        self._create_git_checkpoint("Pre-code-migration checkpoint")
        state = self.migration_agent.apply_openrewrite_recipes(state)
        return state
    
    def _testing_validation_phase(self, state: AgentState) -> AgentState:
        """Execute testing validation phase."""
        print("✅ Testing Validation Phase")
        state["current_phase"] = "testing_validation"
        state = self.validation_agent.validate_migration(state)
        return state
    
    def _performance_validation_phase(self, state: AgentState) -> AgentState:
        """Execute performance validation phase (optional)."""
        print("⚡ Performance Validation Phase")
        state["current_phase"] = "performance_validation"
        
        # This is a placeholder for future performance validation implementation
        print("   • Performance benchmarking...")
        print("   • Memory usage analysis...")
        print("   • Startup time comparison...")
        
        state["last_action_result"] = "performance_validation_success"
        return state
    
    def _final_cleanup_phase(self, state: AgentState) -> AgentState:
        """Execute final cleanup phase."""
        print("🧹 Final Cleanup Phase")
        state["current_phase"] = "final_cleanup"
        
        # Cleanup temporary files, optimize imports, etc.
        print("   • Cleaning up temporary files...")
        print("   • Optimizing imports...")
        print("   • Removing unused dependencies...")
        
        state["last_action_result"] = "cleanup_success"
        return state
    
    def _escalate_phase(self, state: AgentState) -> AgentState:
        """Handle human escalation with phase-specific context."""
        print("🚨 Human Escalation Required")
        state["current_phase"] = "escalation"
        state["escalation_needed"] = True
        
        print(f"\n🔴 ESCALATION: Phase '{state['current_phase']}' requires attention")
        print(f"Error Count: {state['error_count']}")
        print(f"Last Result: {state['last_action_result']}")
        
        # Show phase-specific information
        if hasattr(self, 'deprecation_agent'):
            dep_summary = self.deprecation_agent.get_deprecation_summary()
            print(f"Deprecation items found: {dep_summary['total_deprecation_items']}")
        
        decision = input("\nAction: (c)ontinue, (s)kip phase, (a)bort, (r)econfigure: ").lower()
        
        if decision in ['c', 'continue']:
            state["escalation_needed"] = False
            state["error_count"] = 0
        elif decision in ['s', 'skip']:
            print(f"⏭️ Skipping phase {state['current_phase']}")
            state["escalation_needed"] = False
            state["last_action_result"] = "phase_skipped"
        elif decision in ['r', 'reconfigure']:
            self._reconfigure_phases()
            state["escalation_needed"] = False
        else:
            state["last_action_result"] = "aborted"
        
        return state
    
    def _reconfigure_phases(self):
        """Allow runtime reconfiguration of phases."""
        print("\n🎛️ Phase Reconfiguration")
        print("Current phases:")
        
        for phase, config in self.config.phases.items():
            status = "✅ ENABLED" if config.enabled else "❌ DISABLED"
            print(f"  {phase.value}: {status} (priority: {config.priority})")
        
        while True:
            phase_name = input("\nEnter phase to toggle (or 'done'): ").strip()
            if phase_name.lower() == 'done':
                break
            
            # Find matching phase
            matching_phase = None
            for phase in MigrationPhaseType:
                if phase.value == phase_name or phase.name.lower() == phase_name.lower():
                    matching_phase = phase
                    break
            
            if matching_phase and matching_phase in self.config.phases:
                current_status = self.config.phases[matching_phase].enabled
                self.config.phases[matching_phase].enabled = not current_status
                new_status = "ENABLED" if not current_status else "DISABLED"
                print(f"  {matching_phase.value}: {new_status}")
            else:
                print(f"  Phase '{phase_name}' not found")
    
    def _should_continue(self, state: AgentState) -> str:
        """Enhanced decision logic considering phase configuration."""
        current_phase = state.get("current_phase", "unknown")
        
        # Check phase-specific skip conditions
        if current_phase in self.config.phases:
            phase_config = self.config.phases[MigrationPhaseType(current_phase)]
            if state["error_count"] >= phase_config.retry_count:
                if phase_config.skip_on_failure:
                    print(f"⏭️ Skipping {current_phase} due to configuration")
                    return "continue"
                else:
                    return "escalate"
        
        # Standard continuation logic
        if state["error_count"] >= 3:
            return "escalate"
        elif state["escalation_needed"]:
            return "escalate"
        elif state["last_action_result"] in ["aborted", "validation_success"]:
            return "complete"
        else:
            return "continue"
    
    def _create_git_checkpoint(self, message: str) -> str:
        """Create Git checkpoint with phase context."""
        try:
            repo = git.Repo(self.project_path)
            repo.git.add(A=True)
            commit = repo.index.commit(f"{message} - Session: {self.session_id}")
            print(f"📋 Checkpoint: {commit.hexsha[:8]} - {message}")
            return commit.hexsha
        except Exception as e:
            print(f"⚠️ Checkpoint failed: {e}")
            return ""
    
    def _complete_phase(self, state: AgentState) -> AgentState:
        """Complete migration with comprehensive reporting."""
        print("🎉 Migration Complete")
        state["current_phase"] = "completed"
        
        self._create_git_checkpoint("Migration completed")
        self._generate_comprehensive_report(state)
        
        return state
    
    def _generate_comprehensive_report(self, state: AgentState):
        """Generate comprehensive migration report including deprecation analysis."""
        enabled_phases = [phase.value for phase in self.config.get_enabled_phases()]
        
        # Get deprecation summary if deprecation phase was enabled
        deprecation_summary = {}
        if hasattr(self, 'deprecation_agent'):
            deprecation_summary = self.deprecation_agent.get_deprecation_summary()
        
        report = f"""
# Comprehensive Migration Report - Session {self.session_id}

## Configuration
- **Enabled Phases**: {', '.join(enabled_phases)}
- **Project Path**: {self.project_path}
- **Migration Started**: {datetime.now().isoformat()}
- **Final Status**: {state['last_action_result']}

## Phase Results
{chr(10).join([f"- {msg['content'][:200]}..." for msg in state['messages'][-len(enabled_phases):]])}

## Deprecation Analysis Summary
- **Total deprecation items found**: {deprecation_summary.get('total_deprecation_items', 0)}
- **Automatic fixes applied**: {deprecation_summary.get('auto_fixes_applied', 0)}
- **Items requiring manual review**: {deprecation_summary.get('manual_review_needed', 0)}

## Memory Usage Summary
{chr(10).join([f"- {agent.agent_name}: {agent.memory_manager.get_memory_stats()['usage_percentage']:.1f}%" 
              for agent in [self.analysis_agent, self.migration_agent, self.validation_agent] 
              if hasattr(agent, 'memory_manager')])}

## Recommendations
1. Review deprecation analysis report for manual fixes needed
2. Run comprehensive tests in staging environment
3. Update CI/CD pipeline configurations
4. Monitor application performance post-migration

---
*Generated by Configurable Java Migration System*
        """
        
        report_path = os.path.join(self.project_path, f"comprehensive-migration-report-{self.session_id}.md")
        try:
            with open(report_path, 'w') as f:
                f.write(report)
            print(f"📄 Comprehensive report saved: {report_path}")
        except Exception as e:
            print(f"⚠️ Failed to save report: {e}")
    
    def run_migration(self) -> bool:
        """Execute the complete configurable migration workflow."""
        enabled_phases = self.config.get_enabled_phases()
        
        print(f"🚀 Starting Migration")
        print(f"📂 Project: {self.project_path}")
        print(f"🆔 Session: {self.session_id}")
        print(f"📋 Phases: {[phase.value for phase in enabled_phases]}")
        print("="*80)
        
        # Initialize state
        initial_state = AgentState(
            messages=[],
            project_path=self.project_path,
            session_id=self.session_id,
            current_phase="initialization",
            error_count=0,
            last_action_result="initialized",
            escalation_needed=False
        )
        
        try:
            # Run the configurable LangGraph workflow
            final_state = self.workflow.invoke(initial_state)
            
            success = final_state["last_action_result"] in ["validation_success", "cleanup_success"]
            
            print("\n" + "="*80)
            if success:
                print("🎉 MIGRATION COMPLETED SUCCESSFULLY!")
                print("📋 Key achievements:")
                print("  • Enabled phases executed successfully")
                if hasattr(self, 'deprecation_agent'):
                    dep_summary = self.deprecation_agent.get_deprecation_summary()
                    print(f"  • {dep_summary['total_deprecation_items']} deprecation items analyzed")
                    print(f"  • {dep_summary['auto_fixes_applied']} automatic fixes applied")
            else:
                print(f"⚠️ Migration completed with status: {final_state['last_action_result']}")
            
            print(f"📊 Final Phase: {final_state['current_phase']}")
            print("="*80)
            
            return success
            
        except Exception as e:
            print(f"\n💥 MIGRATION ERROR: {e}")
            import traceback
            traceback.print_exc()
            return False

print("✅ Migration Orchestrator with clean agent references implemented successfully!")

✅ Migration Orchestrator with clean agent references implemented successfully!


In [66]:
# Code Migration Agent with OpenRewrite Integration

class CodeMigrationAgent:
    """Handles code transformations using OpenRewrite recipes."""
    
    def __init__(self, session_id: str):
        self.session_id = session_id
        self.agent_name = "code_migration"
        self.openrewrite_recipes = {
            "java_11_to_17": "org.openrewrite.java.migrate.Java11to17",
            "java_17_to_21": "org.openrewrite.java.migrate.Java17to21",
            "spring_boot_3": "org.openrewrite.java.spring.boot3.UpgradeSpringBoot_3_0",
            "javax_to_jakarta": "org.openrewrite.java.migrate.javax.MigrateJavaxToJakarta",
            "junit4_to_5": "org.openrewrite.java.testing.junit5.JUnit4to5Migration"
        }
        
        # Redis client for logging
        self.redis_client = redis.from_url(REDIS_URL)
    
    def analyze(self, project_path: str) -> Dict[str, Any]:
        """Analyze code for migration opportunities (interface method for orchestrator)."""
        return {
            "applicable_recipes": list(self.openrewrite_recipes.keys()),
            "custom_patterns": [],
            "estimated_changes": 0,
            "success": True,
            "recommendations": [
                "🔧 OpenRewrite recipes available for code transformation",
                f"📝 {len(self.openrewrite_recipes)} transformation recipes ready",
                "🔄 Ready for systematic code migration"
            ]
        }
    
    def log_action(self, action: str, message: str, success: bool = True):
        """Log agent actions to Redis."""
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "agent": self.agent_name,
            "session_id": self.session_id,
            "action": action,
            "message": message,
            "success": success
        }
        log_key = f"agent_log:{self.session_id}:{self.agent_name}"
        self.redis_client.lpush(log_key, json.dumps(log_entry))
        self.redis_client.expire(log_key, 86400)  # 24h expiry
    
    def analyze(self, project_path: str) -> Dict[str, Any]:
        """Analyze code for migration opportunities."""
        return {
            "applicable_recipes": list(self.openrewrite_recipes.keys()),
            "custom_patterns": [],
            "estimated_changes": 0
        }
    
    def execute(self, action: str, context: Dict[str, Any]) -> bool:
        """Execute code migration actions."""
        if action == "apply_recipe":
            recipe_name = context.get("recipe")
            project_path = context.get("project_path")
            return self._apply_openrewrite_recipe(recipe_name, project_path)
        elif action == "migrate_java_version":
            return self._migrate_java_version(context.get("from_version"), context.get("to_version"), context.get("project_path"))
        return False
    
    def _apply_openrewrite_recipe(self, recipe_name: str, project_path: str) -> bool:
        """Apply OpenRewrite recipe to project."""
        if recipe_name not in self.openrewrite_recipes:
            self.log_action("apply_recipe", f"Unknown recipe: {recipe_name}", False)
            return False
        
        recipe_class = self.openrewrite_recipes[recipe_name]
        rewrite_config = f"""
type: specs.openrewrite.org/v1beta/recipe
name: com.migration.{recipe_name}
recipes:
  - {recipe_class}
        """
        
        rewrite_yml_path = os.path.join(project_path, "rewrite.yml")
        
        try:
            with open(rewrite_yml_path, 'w') as f:
                f.write(rewrite_config)
            
            cmd = ["mvn", "org.openrewrite.maven:rewrite-maven-plugin:run", "-f", project_path]
            result = subprocess.run(cmd, capture_output=True, text=True, cwd=project_path)
            
            if result.returncode == 0:
                self.log_action("apply_recipe", f"Successfully applied {recipe_name}")
                return True
            else:
                self.log_action("apply_recipe", f"Failed to apply {recipe_name}: {result.stderr}", False)
                return False
                
        except Exception as e:
            self.log_action("apply_recipe", f"Exception applying {recipe_name}: {str(e)}", False)
            return False
        finally:
            if os.path.exists(rewrite_yml_path):
                os.remove(rewrite_yml_path)
    
    def _migrate_java_version(self, from_version: str, to_version: str, project_path: str) -> bool:
        """Migrate Java version in steps."""
        migration_path = self._get_migration_path(from_version, to_version)
        
        for step in migration_path:
            recipe_name = f"java_{step['from']}_to_{step['to']}"
            if not self._apply_openrewrite_recipe(recipe_name, project_path):
                return False
            
            if not self._update_java_version_in_pom(project_path, step['to']):
                return False
        
        return True
    
    def _get_migration_path(self, from_version: str, to_version: str) -> List[Dict[str, str]]:
        """Get incremental migration path."""
        version_map = {"11": 11, "17": 17, "21": 21}
        from_num = version_map.get(from_version, 11)
        to_num = version_map.get(to_version, 21)
        
        path = []
        current = from_num
        
        while current < to_num:
            if current == 11:
                next_version = 17
            elif current == 17:
                next_version = 21
            else:
                break
                
            path.append({"from": str(current), "to": str(next_version)})
            current = next_version
        
        return path
    
    def _update_java_version_in_pom(self, project_path: str, new_version: str) -> bool:
        """Update Java version in pom.xml."""
        pom_path = os.path.join(project_path, "pom.xml")
        
        try:
            with open(pom_path, 'r') as f:
                content = f.read()
            
            content = re.sub(
                r'<java\.version>.*?</java\.version>',
                f'<java.version>{new_version}</java.version>',
                content
            )
            
            content = re.sub(
                r'<maven\.compiler\.source>.*?</maven\.compiler\.source>',
                f'<maven.compiler.source>{new_version}</maven.compiler.source>',
                content
            )
            content = re.sub(
                r'<maven\.compiler\.target>.*?</maven\.compiler\.target>',
                f'<maven.compiler.target>{new_version}</maven.compiler.target>',
                content
            )
            
            with open(pom_path, 'w') as f:
                f.write(content)
            
            self.log_action("update_java_version", f"Updated to Java {new_version}")
            return True
            
        except Exception as e:
            self.log_action("update_java_version", f"Failed to update pom.xml: {str(e)}", False)
            return False

print("✅ CodeMigrationAgent implemented successfully!")

✅ CodeMigrationAgent implemented successfully!


In [67]:
# Human Interface Agent for MVP Human-in-the-Loop

class HumanInterfaceAgent:
    """Handles human escalation and decision collection."""
    
    def __init__(self, session_id: str):
        self.session_id = session_id
        self.agent_name = "human_interface"
        self.redis_client = redis.from_url(REDIS_URL)
        self.escalation_thresholds = {
            "error_loop_count": 3,
            "no_progress_actions": 5
        }
    
    def analyze(self, project_path: str) -> Dict[str, Any]:
        """Analyze current state for escalation needs (interface method for orchestrator)."""
        return {
            "escalation_needed": False, 
            "success": True,
            "recommendations": [
                "👥 Human interface ready for escalations",
                "🚨 No immediate human intervention needed",
                "📋 Decision tracking system active"
            ]
        }
    
    def log_action(self, action: str, message: Any, success: bool = True):
        """Log agent actions to Redis."""
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "agent": self.agent_name,
            "session_id": self.session_id,
            "action": action,
            "message": str(message),
            "success": success
        }
        log_key = f"agent_log:{self.session_id}:{self.agent_name}"
        self.redis_client.lpush(log_key, json.dumps(log_entry))
        self.redis_client.expire(log_key, 86400)  # 24h expiry
    
    def analyze(self, project_path: str) -> Dict[str, Any]:
        """Analyze current state for escalation needs."""
        return {"escalation_needed": False}
    
    def should_escalate(self, context: Dict[str, Any]) -> bool:
        """Determine if human escalation is needed."""
        if context.get("consecutive_errors", 0) >= self.escalation_thresholds["error_loop_count"]:
            return True
            
        if context.get("actions_without_progress", 0) >= self.escalation_thresholds["no_progress_actions"]:
            return True
            
        risk_level = context.get("risk_level", RiskLevel.LOW)
        if risk_level in [RiskLevel.HIGH, RiskLevel.CRITICAL]:
            return True
            
        return False
    
    def create_escalation(self, escalation_context: EscalationContext) -> Dict[str, Any]:
        """Create human escalation request."""
        escalation_data = {
            "escalation_type": escalation_context.escalation_type,
            "context": escalation_context.context,
            "recommendation": escalation_context.recommendation,
            "risk_level": escalation_context.risk_level.value,
            "file_path": escalation_context.file_path,
            "current_code": escalation_context.current_code,
            "options": escalation_context.options or [],
            "timestamp": datetime.now().isoformat(),
            "session_id": self.session_id
        }
        
        escalation_key = f"escalation:{self.session_id}:{datetime.now().isoformat()}"
        self.redis_client.set(escalation_key, json.dumps(escalation_data))
        
        self.log_action("escalation_created", escalation_data)
        return escalation_data
    
    def get_human_decision(self, escalation_id: str) -> Optional[Decision]:
        """Retrieve human decision for escalation (blocking call in MVP)."""
        print(f"\n🚨 HUMAN INTERVENTION REQUIRED 🚨")
        print(f"Escalation ID: {escalation_id}")
        
        escalation_data = json.loads(self.redis_client.get(escalation_id) or "{}")
        
        print(f"\nType: {escalation_data.get('escalation_type')}")
        print(f"Risk Level: {escalation_data.get('risk_level')}")
        print(f"File: {escalation_data.get('file_path', 'N/A')}")
        print(f"Context: {escalation_data.get('context')}")
        
        options = escalation_data.get('options', [])
        if options:
            print("\nAvailable Options:")
            for i, option in enumerate(options):
                print(f"{i+1}. {option.get('description', 'No description')}")
                print(f"   Confidence: {option.get('confidence', 0):.1%}")
                print(f"   Impact: {option.get('impact', 'Unknown')}")
        
        choice = input("\nEnter your choice (number) or 'skip': ")
        rationale = input("Rationale for decision: ")
        
        if choice.lower() == 'skip':
            return None
            
        try:
            choice_idx = int(choice) - 1
            chosen_option = options[choice_idx]['id'] if 0 <= choice_idx < len(options) else choice
        except (ValueError, IndexError):
            chosen_option = choice
        
        decision = Decision(
            timestamp=datetime.now().isoformat(),
            context=escalation_data.get('context', {}),
            file_path=escalation_data.get('file_path', ''),
            issue=escalation_data.get('escalation_type', ''),
            options=options,
            chosen_option=chosen_option,
            rationale=rationale,
            risk_level=RiskLevel(escalation_data.get('risk_level', 'low'))
        )
        
        decision_key = f"decision:{self.session_id}:{escalation_id}"
        self.redis_client.set(decision_key, decision.json())
        
        self.log_action("human_decision_recorded", decision.dict())
        return decision
    
    def execute(self, action: str, context: Dict[str, Any]) -> bool:
        """Execute human interface actions."""
        if action == "escalate":
            escalation_context = context.get("escalation_context")
            if escalation_context:
                self.create_escalation(escalation_context)
                return True
        return False

print("✅ HumanInterfaceAgent implemented successfully!")

✅ HumanInterfaceAgent implemented successfully!


In [68]:
# Migration Orchestrator - Main coordination agent

class MigrationOrchestrator:
    """Main orchestrator that coordinates all migration agents."""
    
    def __init__(self, project_path: str, session_id: str = None, config=None):
        self.project_path = project_path
        self.session_id = session_id or f"migration_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        self.redis_client = redis.from_url(REDIS_URL)
        
        # Use provided configuration or create default (will be None for now)
        self.config = config
        
        # Initialize agents with direct memory sharing (MVP approach)
        self.agents = {
            "analysis": AnalysisAgent(self.session_id),
            "code_migration": CodeMigrationAgent(self.session_id),
            "human_interface": HumanInterfaceAgent(self.session_id)
        }
        
        # Migration state
        self.state = self._initialize_state()
        self.consecutive_errors = 0
        self.actions_without_progress = 0
        
    def _initialize_state(self) -> MigrationState:
        """Initialize or load migration state."""
        state_key = f"migration_state:{self.session_id}"
        existing_state = self.redis_client.get(state_key)
        
        if existing_state:
            return MigrationState.parse_raw(existing_state)
        
        state = MigrationState(
            project_path=self.project_path,
            current_phase=MigrationPhase.ANALYSIS,
            completed_steps=[],
            failed_attempts={},
            checkpoints=[],
            human_decisions=[],
            risk_assessment={},
            session_id=self.session_id,
            started_at=datetime.now().isoformat(),
            last_updated=datetime.now().isoformat()
        )
        
        self._save_state(state)
        return state
    
    def _save_state(self, state: MigrationState):
        """Save migration state to Redis."""
        state.last_updated = datetime.now().isoformat()
        state_key = f"migration_state:{self.session_id}"
        self.redis_client.set(state_key, state.json(), ex=86400)  # 24h expiry
    
    def create_checkpoint(self) -> str:
        """Create Git checkpoint for current state."""
        try:
            repo = git.Repo(self.project_path)
            repo.git.add(A=True)
            commit_msg = f"Migration checkpoint - {self.state.current_phase.value}"
            commit = repo.index.commit(commit_msg)
            
            self.state.checkpoints.append(commit.hexsha)
            self._save_state(self.state)
            
            print(f"✅ Checkpoint created: {commit.hexsha[:8]}")
            return commit.hexsha
            
        except Exception as e:
            print(f"❌ Failed to create checkpoint: {e}")
            return ""
    
    def migrate_project(self) -> bool:
        """Execute full project migration workflow."""
        print(f"🚀 Starting migration for project: {self.project_path}")
        print(f"📋 Session ID: {self.session_id}")
        
        try:
            phases = [
                MigrationPhase.ANALYSIS,
                MigrationPhase.JAVA_VERSION_UPGRADE,
                MigrationPhase.SPRING_BOOT_UPGRADE,
                MigrationPhase.JAVAX_TO_JAKARTA,
                MigrationPhase.JUNIT_MIGRATION
            ]
            
            for phase in phases:
                if not self._execute_phase(phase):
                    return False
                
            self.state.current_phase = MigrationPhase.COMPLETED
            self.state.completed_steps.append("migration_completed")
            self._save_state(self.state)
            
            print("🎉 Migration completed successfully!")
            return True
            
        except Exception as e:
            print(f"💥 Migration failed with exception: {e}")
            return False

print("✅ MigrationOrchestrator base implemented successfully - now accepts config parameter!")

✅ MigrationOrchestrator base implemented successfully - now accepts config parameter!


In [69]:
# Phase execution methods for MigrationOrchestrator

def _execute_phase(self, phase: MigrationPhase) -> bool:
    """Execute a specific migration phase."""
    print(f"\n📍 Executing phase: {phase.value}")
    self.state.current_phase = phase
    self._save_state(self.state)
    
    success = False
    max_retries = 3
    
    for attempt in range(max_retries):
        try:
            if phase == MigrationPhase.ANALYSIS:
                success = self._run_analysis_phase()
            elif phase == MigrationPhase.JAVA_VERSION_UPGRADE:
                success = self._run_java_upgrade_phase()
            elif phase == MigrationPhase.SPRING_BOOT_UPGRADE:
                success = self._run_spring_boot_phase()
            elif phase == MigrationPhase.JAVAX_TO_JAKARTA:
                success = self._run_javax_jakarta_phase()
            elif phase == MigrationPhase.JUNIT_MIGRATION:
                success = self._run_junit_phase()
            
            if success:
                self.consecutive_errors = 0
                self.actions_without_progress = 0
                self.state.completed_steps.append(phase.value)
                self.create_checkpoint()
                print(f"✅ Phase {phase.value} completed successfully")
                return True
            else:
                self.consecutive_errors += 1
                self.actions_without_progress += 1
                
        except Exception as e:
            print(f"❌ Phase {phase.value} failed on attempt {attempt + 1}: {e}")
            self.consecutive_errors += 1
            
            if self._should_escalate():
                if self._handle_escalation(phase, str(e)):
                    success = True
                    break
    
    if not success:
        phase_key = phase.value
        if phase_key not in self.state.failed_attempts:
            self.state.failed_attempts[phase_key] = []
        self.state.failed_attempts[phase_key].append(f"Failed after {max_retries} attempts")
        self._save_state(self.state)
        
    return success

def _run_analysis_phase(self) -> bool:
    """Execute project analysis phase."""
    try:
        print("🔍 Analyzing project structure...")
        analysis_results = self.agents["analysis"].analyze(self.project_path)
        
        self.state.risk_assessment = analysis_results.get("risk_assessment", {})
        self._save_state(self.state)
        
        report_success = self.agents["analysis"].execute("generate_report", analysis_results)
        
        print(f"📊 Analysis completed:")
        print(f"   Java Version: {analysis_results.get('java_version')}")
        print(f"   Spring Boot: {analysis_results.get('spring_boot_version')}")
        print(f"   Deprecated Components: {len(analysis_results.get('deprecated_components', []))}")
        
        return report_success
        
    except Exception as e:
        print(f"❌ Analysis failed: {e}")
        return False

def _run_java_upgrade_phase(self) -> bool:
    """Execute Java version upgrade phase."""
    try:
        print("☕ Upgrading Java version...")
        
        analysis_agent = self.agents["analysis"]
        current_version = analysis_agent._detect_java_version(self.project_path)
        target_version = "21"
        
        if current_version == target_version:
            print(f"✅ Java version already at {target_version}")
            return True
            
        success = self.agents["code_migration"].execute("migrate_java_version", {
            "from_version": current_version,
            "to_version": target_version,
            "project_path": self.project_path
        })
        
        if success:
            print(f"✅ Java upgraded from {current_version} to {target_version}")
        
        return success
        
    except Exception as e:
        print(f"❌ Java upgrade failed: {e}")
        return False

def _run_spring_boot_phase(self) -> bool:
    """Execute Spring Boot upgrade phase."""
    try:
        print("🍃 Upgrading Spring Boot...")
        
        success = self.agents["code_migration"].execute("apply_recipe", {
            "recipe": "spring_boot_3",
            "project_path": self.project_path
        })
        
        if success:
            print("✅ Spring Boot upgraded to 3.x")
        
        return success
        
    except Exception as e:
        print(f"❌ Spring Boot upgrade failed: {e}")
        return False

def _run_javax_jakarta_phase(self) -> bool:
    """Execute javax to jakarta migration phase."""
    try:
        print("📦 Migrating javax to jakarta...")
        
        success = self.agents["code_migration"].execute("apply_recipe", {
            "recipe": "javax_to_jakarta",
            "project_path": self.project_path
        })
        
        if success:
            print("✅ javax → jakarta migration completed")
        
        return success
        
    except Exception as e:
        print(f"❌ javax → jakarta migration failed: {e}")
        return False

def _run_junit_phase(self) -> bool:
    """Execute JUnit 4 to 5 migration phase."""
    try:
        print("🧪 Migrating JUnit 4 to 5...")
        
        success = self.agents["code_migration"].execute("apply_recipe", {
            "recipe": "junit4_to_5",
            "project_path": self.project_path
        })
        
        if success:
            print("✅ JUnit 4 → 5 migration completed")
        
        return success
        
    except Exception as e:
        print(f"❌ JUnit migration failed: {e}")
        return False

def _should_escalate(self) -> bool:
    """Determine if human escalation is needed."""
    return self.agents["human_interface"].should_escalate({
        "consecutive_errors": self.consecutive_errors,
        "actions_without_progress": self.actions_without_progress,
        "risk_level": RiskLevel.HIGH if self.consecutive_errors >= 2 else RiskLevel.LOW
    })

def _handle_escalation(self, phase: MigrationPhase, error_msg: str) -> bool:
    """Handle human escalation for a failed phase."""
    escalation_context = EscalationContext(
        escalation_type="phase_failure",
        context={
            "phase": phase.value,
            "error": error_msg,
            "consecutive_errors": self.consecutive_errors,
            "failed_attempts": self.state.failed_attempts.get(phase.value, [])
        },
        recommendation="skip_phase",
        risk_level=RiskLevel.HIGH,
        options=[
            {
                "id": "retry",
                "description": "Retry the phase with manual intervention",
                "confidence": 0.3,
                "impact": "May resolve temporary issues"
            },
            {
                "id": "skip",
                "description": "Skip this phase and continue",
                "confidence": 0.7,
                "impact": "Phase will remain incomplete"
            },
            {
                "id": "abort",
                "description": "Abort entire migration",
                "confidence": 1.0,
                "impact": "Migration will be cancelled"
            }
        ]
    )
    
    escalation_data = self.agents["human_interface"].create_escalation(escalation_context)
    escalation_id = f"escalation:{self.session_id}:{datetime.now().isoformat()}"
    
    decision = self.agents["human_interface"].get_human_decision(escalation_id)
    
    if decision:
        self.state.human_decisions.append(decision)
        self._save_state(self.state)
        
        if decision.chosen_option == "retry":
            return False
        elif decision.chosen_option == "skip":
            print(f"⏭️  Skipping phase {phase.value} per human decision")
            return True
        elif decision.chosen_option == "abort":
            print(f"🛑 Aborting migration per human decision")
            raise Exception("Migration aborted by human decision")
    
    return False

# Add methods to MigrationOrchestrator class
MigrationOrchestrator._execute_phase = _execute_phase
MigrationOrchestrator._run_analysis_phase = _run_analysis_phase
MigrationOrchestrator._run_java_upgrade_phase = _run_java_upgrade_phase
MigrationOrchestrator._run_spring_boot_phase = _run_spring_boot_phase
MigrationOrchestrator._run_javax_jakarta_phase = _run_javax_jakarta_phase
MigrationOrchestrator._run_junit_phase = _run_junit_phase
MigrationOrchestrator._should_escalate = _should_escalate
MigrationOrchestrator._handle_escalation = _handle_escalation

print("✅ Phase implementation methods added to MigrationOrchestrator!")

✅ Phase implementation methods added to MigrationOrchestrator!


In [70]:
# Updated Demo Functions with Clean References

def create_enhanced_test_project() -> str:
    """Create a test Java 11 project for demonstration."""
    import tempfile
    import textwrap
    
    project_dir = tempfile.mkdtemp(prefix="java_migration_test_")
    
    # Create Maven directory structure
    src_main_java = os.path.join(project_dir, "src", "main", "java", "com", "example")
    src_test_java = os.path.join(project_dir, "src", "test", "java", "com", "example")
    os.makedirs(src_main_java, exist_ok=True)
    os.makedirs(src_test_java, exist_ok=True)
    
    # Create pom.xml with Java 11 and deprecated dependencies
    pom_content = textwrap.dedent("""
    <?xml version="1.0" encoding="UTF-8"?>
    <project xmlns="http://maven.apache.org/POM/4.0.0"
             xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
             xsi:schemaLocation="http://maven.apache.org/POM/4.0.0 
                                 http://maven.apache.org/xsd/maven-4.0.0.xsd">
        <modelVersion>4.0.0</modelVersion>
        
        <groupId>com.example</groupId>
        <artifactId>java-migration-test</artifactId>
        <version>1.0.0</version>
        <packaging>jar</packaging>
        
        <properties>
            <maven.compiler.source>11</maven.compiler.source>
            <maven.compiler.target>11</maven.compiler.target>
            <java.version>11</java.version>
            <spring.boot.version>2.7.0</spring.boot.version>
            <junit.version>4.13.2</junit.version>
        </properties>
        
        <dependencies>
            <!-- Spring Boot 2.x (needs upgrade to 3.x) -->
            <dependency>
                <groupId>org.springframework.boot</groupId>
                <artifactId>spring-boot-starter-web</artifactId>
                <version>${spring.boot.version}</version>
            </dependency>
            
            <!-- JUnit 4 (needs migration to 5) -->
            <dependency>
                <groupId>junit</groupId>
                <artifactId>junit</artifactId>
                <version>${junit.version}</version>
                <scope>test</scope>
            </dependency>
            
            <!-- javax dependencies (need jakarta migration) -->
            <dependency>
                <groupId>javax.servlet</groupId>
                <artifactId>javax.servlet-api</artifactId>
                <version>4.0.1</version>
            </dependency>
        </dependencies>
        
        <build>
            <plugins>
                <plugin>
                    <groupId>org.apache.maven.plugins</groupId>
                    <artifactId>maven-compiler-plugin</artifactId>
                    <version>3.8.1</version>
                    <configuration>
                        <source>11</source>
                        <target>11</target>
                    </configuration>
                </plugin>
                
                <plugin>
                    <groupId>org.apache.maven.plugins</groupId>
                    <artifactId>maven-surefire-plugin</artifactId>
                    <version>2.22.2</version>
                </plugin>
            </plugins>
        </build>
    </project>
    """)
    
    with open(os.path.join(project_dir, "pom.xml"), 'w') as f:
        f.write(pom_content)
    
    # Create Java source with deprecated patterns
    java_content = textwrap.dedent("""
    package com.example;
    
    import javax.servlet.http.HttpServlet;
    import javax.servlet.http.HttpServletRequest;
    import javax.servlet.http.HttpServletResponse;
    
    public class TestService {
        
        public void doSomething() {
            // Deprecated constructor usage
            Integer number = new Integer(42);
            Double value = new Double(3.14);
            
            System.out.println("Number: " + number);
            System.out.println("Value: " + value);
        }
        
        @Deprecated
        public void oldMethod() {
            System.out.println("This is a deprecated method");
        }
    }
    """)
    
    with open(os.path.join(src_main_java, "TestService.java"), 'w') as f:
        f.write(java_content)
    
    # Create JUnit 4 test
    test_content = textwrap.dedent("""
    package com.example;
    
    import org.junit.Test;
    import org.junit.Before;
    import org.junit.Assert;
    
    public class TestServiceTest {
        
        private TestService service;
        
        @Before
        public void setUp() {
            service = new TestService();
        }
        
        @Test
        public void testDoSomething() {
            service.doSomething();
            Assert.assertTrue("Service should work", true);
        }
        
        @Test
        public void testOldMethod() {
            service.oldMethod();
            Assert.assertNotNull("Service should not be null", service);
        }
    }
    """)
    
    with open(os.path.join(src_test_java, "TestServiceTest.java"), 'w') as f:
        f.write(test_content)
    
    print(f"📁 Enhanced test project created at: {project_dir}")
    print("   • Java 11 with deprecated constructor usage")
    print("   • Spring Boot 2.7.0 (needs upgrade to 3.x)")
    print("   • JUnit 4 tests (need migration to 5)")
    print("   • javax.servlet dependencies (need jakarta migration)")
    print("   • Maven plugins with old versions")
    
    return project_dir

def run_migration_demo(project_path: str = None):
    """Run the complete migration demonstration with clean agent references."""
    
    if not project_path:
        project_path = create_enhanced_test_project()
    
    print("\n" + "="*80)
    print("🚀 ENHANCED JAVA MIGRATION SYSTEM WITH CONFIGURABLE PHASES")
    print("="*80)
    print("🔧 Features:")
    print("  • Configurable migration phases (enable/disable any phase)")
    print("  • Advanced deprecation detection with LLM analysis")
    print("  • Automatic replacement of deprecated APIs/patterns")
    print("  • Maven plugin compatibility checking")
    print("  • Large context memory (200k tokens)")
    print("  • File-level locking for concurrent operations")
    print("  • Comprehensive error analysis and web research")
    print("="*80)
    
    try:
        # Create default configuration
        config = MigrationConfiguration()
        orchestrator = MigrationOrchestrator(project_path, config=config)
        
        print(f"\n📂 Project Path: {project_path}")
        print(f"🆔 Session ID: {orchestrator.session_id}")
        print(f"🔗 Redis URL: {REDIS_URL}")
        print(f"🔍 Bing Search: {'✅ Configured' if BING_SEARCH_API_KEY != 'your-bing-api-key' else '❌ Not configured'}")
        
        enabled_phases = config.get_enabled_phases()
        print(f"📋 Enabled Phases: {[phase.value for phase in enabled_phases]}")
        
        success = orchestrator.migrate_project()
        
        if success:
            print("\n🎉 MIGRATION COMPLETED SUCCESSFULLY!")
            print("📋 Key achievements:")
            print("  • All enabled phases executed successfully")
            if hasattr(orchestrator, 'deprecation_agent'):
                dep_summary = orchestrator.deprecation_agent.get_deprecation_summary()
                print(f"  • {dep_summary['total_deprecation_items']} deprecation items analyzed")
                print(f"  • {dep_summary['auto_fixes_applied']} automatic fixes applied")
                print(f"  • {dep_summary['manual_review_needed']} items need manual review")
        else:
            print("\n⚠️ MIGRATION COMPLETED WITH ISSUES")
            print("📋 Check the comprehensive migration report for details")
    
    except Exception as e:
        print(f"\n💥 MIGRATION SYSTEM ERROR: {e}")
        import traceback
        traceback.print_exc()
    
    print("\n" + "="*80)

def run_configurable_migration_demo(project_path: str = None, interactive_config: bool = True):
    """Run migration with configurable phases including deprecation detection."""
    
    if not project_path:
        project_path = create_enhanced_test_project()
    
    print("\n" + "="*80)
    print("🎛️ CONFIGURABLE JAVA MIGRATION SYSTEM")
    print("="*80)
    print("🆕 NEW FEATURES:")
    print("  • Configurable migration phases")
    print("  • Advanced deprecation detection with LLM analysis")
    print("  • Automatic replacement of deprecated APIs/patterns")
    print("  • Maven plugin deprecation scanning")
    print("  • Phase-specific error handling and retry logic")
    print("  • Runtime phase reconfiguration")
    print("="*80)
    
    # Create configuration
    if interactive_config:
        config = create_custom_migration_config()
    else:
        # Default configuration with all phases enabled
        config = MigrationConfiguration()
        print("📋 Using default configuration with all phases enabled")
    
    try:
        # Create orchestrator with custom configuration
        orchestrator = MigrationOrchestrator(project_path, config=config)
        
        print(f"\n📂 Project Path: {project_path}")
        print(f"🆔 Session ID: {orchestrator.session_id}")
        print(f"🔗 Redis URL: {REDIS_URL}")
        
        enabled_phases = config.get_enabled_phases()
        print(f"📋 Enabled Phases: {[phase.value for phase in enabled_phases]}")
        
        # Show deprecation detection config if enabled
        if MigrationPhaseType.DEPRECATION_DETECTION in enabled_phases:
            dep_config = config.phases[MigrationPhaseType.DEPRECATION_DETECTION]
            print(f"🔍 Deprecation Detection:")
            if dep_config.custom_params:
                for key, value in dep_config.custom_params.items():
                    print(f"   • {key}: {value}")
        
        success = orchestrator.migrate_project()
        
        if success:
            print("\n🎉 CONFIGURABLE MIGRATION COMPLETED SUCCESSFULLY!")
            print("📋 Key achievements:")
            
            # Show deprecation-specific results
            if hasattr(orchestrator, 'deprecation_agent'):
                dep_summary = orchestrator.deprecation_agent.get_deprecation_summary()
                print(f"  • {dep_summary['total_deprecation_items']} deprecation items analyzed")
                print(f"  • {dep_summary['auto_fixes_applied']} automatic fixes applied")
                print(f"  • {dep_summary['manual_review_needed']} items need manual review")
                
                if dep_summary['deprecation_categories']:
                    print("  • Categories found:")
                    for category, items in dep_summary['deprecation_categories'].items():
                        if items:
                            print(f"    - {category}: {len(items)} items")
        else:
            print("\n⚠️ MIGRATION COMPLETED WITH ISSUES")
            print("📋 Check the comprehensive migration report for details")
    
    except Exception as e:
        print(f"\n💥 CONFIGURABLE MIGRATION ERROR: {e}")
        import traceback
        traceback.print_exc()
    
    print("\n" + "="*80)

def demo_deprecation_only(project_path: str = None):
    """Demo just the deprecation detection phase."""
    
    if not project_path:
        project_path = create_enhanced_test_project()
    
    print("\n🔍 DEPRECATION DETECTION DEMO")
    print("="*50)
    
    # Create config with only deprecation detection enabled
    config = MigrationConfiguration()
    
    # Disable all phases except analysis and deprecation detection
    for phase in MigrationPhaseType:
        config.disable_phase(phase)
    
    config.enable_phase(MigrationPhaseType.ANALYSIS)
    config.enable_phase(MigrationPhaseType.DEPRECATION_DETECTION, 
                       custom_params={
                           "deep_scan": True,
                           "include_transitive_deps": True,
                           "check_plugin_compatibility": True,
                           "analyze_code_patterns": True,
                           "auto_apply_safe_fixes": True
                       })
    
    try:
        orchestrator = MigrationOrchestrator(project_path, config=config)
        success = orchestrator.migrate_project()
        
        if success and hasattr(orchestrator, 'deprecation_agent'):
            print("\n📊 Deprecation Detection Results:")
            summary = orchestrator.deprecation_agent.get_deprecation_summary()
            
            print(f"   • Total items found: {summary['total_deprecation_items']}")
            print(f"   • Auto-fixes applied: {summary['auto_fixes_applied']}")
            print(f"   • Manual review needed: {summary['manual_review_needed']}")
            
            print("\n📁 Generated Reports:")
            print(f"   • Deprecation analysis report")
            print(f"   • Comprehensive migration report")
            
    except Exception as e:
        print(f"❌ Demo failed: {e}")

# Create placeholder for missing create_custom_migration_config function
def create_custom_migration_config() -> MigrationConfiguration:
    """Create a custom migration configuration with user preferences."""
    config = MigrationConfiguration()
    
    print("🎛️ Custom Migration Configuration")
    print("="*50)
    print("Using default configuration for now...")
    
    return config

# Keep the rest of the utility functions the same but update usage examples
print("✅ Updated demo functions with clean agent references implemented!")
print("\n📚 Usage Examples:")
print("1. run_migration_demo()  # Default migration with all phases")
print("2. run_configurable_migration_demo()  # Interactive phase configuration")
print("3. demo_deprecation_only()  # Test deprecation detection only")
print("4. test_deprecation_tools_individually()  # Test individual tools")
print("5. monitor_memory_usage('session_id')  # Monitor memory usage")
print("\n✅ Clean Class Names:")
print("   • AnalysisAgent (with enhanced 200k memory)")
print("   • CodeMigrationAgent (with comprehensive tools)")
print("   • ValidationAgent (with AI-powered analysis)")  
print("   • DeprecationAgent (with LLM-powered detection)")
print("   • MigrationOrchestrator (with configurable phases)")
print("\n🎛️ All references are now consistent throughout the system!")

✅ Updated demo functions with clean agent references implemented!

📚 Usage Examples:
1. run_migration_demo()  # Default migration with all phases
2. run_configurable_migration_demo()  # Interactive phase configuration
3. demo_deprecation_only()  # Test deprecation detection only
4. test_deprecation_tools_individually()  # Test individual tools
5. monitor_memory_usage('session_id')  # Monitor memory usage

✅ Clean Class Names:
   • AnalysisAgent (with enhanced 200k memory)
   • CodeMigrationAgent (with comprehensive tools)
   • ValidationAgent (with AI-powered analysis)
   • DeprecationAgent (with LLM-powered detection)
   • MigrationOrchestrator (with configurable phases)

🎛️ All references are now consistent throughout the system!


In [71]:
# Configuration and Demonstration Functions

def create_custom_migration_config() -> MigrationConfiguration:
    """Create a custom migration configuration with user preferences."""
    config = MigrationConfiguration()
    
    print("🎛️ Custom Migration Configuration")
    print("="*50)
    
    # Show current configuration
    print("Current phase configuration:")
    for phase, phase_config in config.phases.items():
        status = "✅ ENABLED" if phase_config.enabled else "❌ DISABLED"
        print(f"  {phase.value}: {status} (priority: {phase_config.priority})")
    
    print("\nWould you like to modify any phases? (y/n)")
    modify_response = input().lower() if hasattr(__builtins__, 'input') else 'n'
    
    if modify_response.startswith('y'):
        for phase in MigrationPhaseType:
            current = config.phases[phase].enabled
            print(f"\n{phase.value} - Currently: {'ENABLED' if current else 'DISABLED'}")
            response = input(f"Enable {phase.value}? (y/n/skip): ").lower() if hasattr(__builtins__, 'input') else 'skip'
            
            if response.startswith('y'):
                config.enable_phase(phase)
            elif response.startswith('n'):
                config.disable_phase(phase)
    
    # Configure deprecation detection specifics
    if config.phases[MigrationPhaseType.DEPRECATION_DETECTION].enabled:
        print("\n🔍 Deprecation Detection Configuration:")
        print("1. Deep scan (thorough but slower)")
        print("2. Quick scan (faster but less comprehensive)")
        choice = input("Choose scan type (1/2): ") if hasattr(__builtins__, 'input') else "1"
        
        deep_scan = choice == "1"
        config.update_phase_config(
            MigrationPhaseType.DEPRECATION_DETECTION,
            custom_params={
                "deep_scan": deep_scan,
                "include_transitive_deps": deep_scan,
                "check_plugin_compatibility": True,
                "analyze_code_patterns": deep_scan,
                "auto_apply_safe_fixes": True
            }
        )
    
    return config

def show_phase_configuration_examples():
    """Show examples of different phase configurations."""
    print("\n📚 Phase Configuration Examples")
    print("="*50)
    
    examples = [
        {
            "name": "Quick Migration (Essential phases only)",
            "enabled": [MigrationPhaseType.ANALYSIS, MigrationPhaseType.DEPENDENCY_UPDATE, MigrationPhaseType.CODE_MIGRATION],
            "description": "Fast migration focusing on core changes"
        },
        {
            "name": "Comprehensive Migration (All phases)",
            "enabled": list(MigrationPhaseType),
            "description": "Complete migration with all analysis and validation"
        },
        {
            "name": "Deprecation Analysis Only",
            "enabled": [MigrationPhaseType.ANALYSIS, MigrationPhaseType.DEPRECATION_DETECTION],
            "description": "Focus on identifying and fixing deprecated usage"
        },
        {
            "name": "Code Quality Focus",
            "enabled": [MigrationPhaseType.ANALYSIS, MigrationPhaseType.DEPRECATION_DETECTION, 
                       MigrationPhaseType.CODE_MIGRATION, MigrationPhaseType.FINAL_CLEANUP],
            "description": "Modernization with cleanup and optimization"
        }
    ]
    
    for i, example in enumerate(examples, 1):
        print(f"\n{i}. {example['name']}")
        print(f"   Description: {example['description']}")
        print(f"   Phases: {[phase.value for phase in example['enabled']]}")
        
        # Show how to configure
        print(f"   Configuration:")
        print(f"   ```python")
        print(f"   config = MigrationConfiguration()")
        print(f"   # Disable all phases first")
        print(f"   for phase in MigrationPhaseType:")
        print(f"       config.disable_phase(phase)")
        print(f"   # Enable desired phases")
        for phase in example['enabled']:
            print(f"   config.enable_phase(MigrationPhaseType.{phase.name})")
        print(f"   ```")

def test_deprecation_tools_individually():
    """Test the deprecation detection tools individually."""
    print("\n🧪 Testing Deprecation Detection Tools")
    print("="*50)
    
    # Create a test project
    test_project = create_enhanced_test_project()
    
    print("1. Testing Maven deprecation analysis...")
    try:
        result = run_maven_with_deprecation_analysis(test_project)
        print(f"✅ Maven analysis result: {result[:200]}...")
    except Exception as e:
        print(f"❌ Maven analysis failed: {e}")
    
    print("\n2. Testing plugin deprecation scanning...")
    try:
        result = scan_maven_plugins_deprecation(test_project)
        print(f"✅ Plugin scan result: {result[:200]}...")
    except Exception as e:
        print(f"❌ Plugin scan failed: {e}")
    
    print("\n3. Testing LLM deprecation analysis...")
    try:
        sample_warnings = "warning: [deprecation] new Integer(int) is deprecated"
        result = analyze_deprecated_apis_with_llm(test_project, sample_warnings)
        print(f"✅ LLM analysis result: {result[:200]}...")
    except Exception as e:
        print(f"❌ LLM analysis failed: {e}")
    
    print("\n4. Testing automatic replacements...")
    try:
        sample_analysis = "deprecated items found: new Integer() usage"
        result = auto_replace_simple_deprecations(test_project, sample_analysis)
        print(f"✅ Auto replacement result: {result[:200]}...")
    except Exception as e:
        print(f"❌ Auto replacement failed: {e}")
    
    print("\n✅ Individual tool testing completed!")

print("✅ Configuration and demonstration functions implemented!")
print("\n📚 New Usage Examples:")
print("1. run_configurable_migration_demo()  # Interactive configuration")
print("2. run_configurable_migration_demo('/path/to/project', False)  # Default config")
print("3. demo_deprecation_only()  # Test deprecation detection only")
print("4. show_phase_configuration_examples()  # See configuration examples")
print("5. test_deprecation_tools_individually()  # Test individual tools")
print("\n🎛️ Key Features:")
print("   • Configurable migration phases (enable/disable any phase)")
print("   • Advanced deprecation detection with LLM analysis")
print("   • Automatic replacement of deprecated patterns")
print("   • Maven plugin compatibility checking")
print("   • Interactive phase configuration")
print("   • Runtime phase reconfiguration during migration")
print("   • Phase-specific error handling and retry logic")
print("   • Comprehensive reporting with deprecation analysis")

✅ Configuration and demonstration functions implemented!

📚 New Usage Examples:
1. run_configurable_migration_demo()  # Interactive configuration
2. run_configurable_migration_demo('/path/to/project', False)  # Default config
3. demo_deprecation_only()  # Test deprecation detection only
4. show_phase_configuration_examples()  # See configuration examples
5. test_deprecation_tools_individually()  # Test individual tools

🎛️ Key Features:
   • Configurable migration phases (enable/disable any phase)
   • Advanced deprecation detection with LLM analysis
   • Automatic replacement of deprecated patterns
   • Maven plugin compatibility checking
   • Interactive phase configuration
   • Runtime phase reconfiguration during migration
   • Phase-specific error handling and retry logic
   • Comprehensive reporting with deprecation analysis


In [72]:
# Memory Monitoring and Testing Utilities

def monitor_memory_usage(session_id: str):
    """Monitor memory usage across all agents in a session."""
    redis_client = redis.from_url(REDIS_URL)
    agents = ["analysis", "code_migration", "validation"]
    
    print(f"📊 Memory Usage Report for Session: {session_id}")
    print("="*70)
    
    total_tokens = 0
    total_summarizations = 0
    
    for agent_name in agents:
        memory_key = f"memory_stats:{session_id}:{agent_name}"
        stats_data = redis_client.get(memory_key)
        
        if stats_data:
            stats = json.loads(stats_data)
            total_tokens += stats.get('estimated_tokens', 0)
            total_summarizations += stats.get('summarization_count', 0)
            
            print(f"🤖 {agent_name.title()} Agent:")
            print(f"   • Tokens: {stats.get('estimated_tokens', 0):,}/{stats.get('max_tokens', 0):,}")
            print(f"   • Usage: {stats.get('usage_percentage', 0):.1f}%")
            print(f"   • Messages: {stats.get('messages_count', 0)}")
            print(f"   • Summarizations: {stats.get('summarization_count', 0)}")
            print(f"   • Last Update: {stats.get('timestamp', 'N/A')}")
            
            # Check for summarization events
            event_key = f"summarization_log:{session_id}:{agent_name}"
            events = redis_client.lrange(event_key, 0, -1)
            if events:
                print(f"   • Summarization Events: {len(events)}")
                for event_data in events[:3]:  # Show last 3 events
                    event = json.loads(event_data)
                    print(f"     - #{event['summarization_number']}: {event['messages_summarized']} msgs → {event['summary_length']} chars")
        else:
            print(f"🤖 {agent_name.title()} Agent: No memory data found")
        
        print()
    
    print(f"📈 Total Session Summary:")
    print(f"   • Combined Token Usage: {total_tokens:,}")
    print(f"   • Total Summarizations: {total_summarizations}")
    print("="*70)

def test_memory_system():
    """Test the enhanced memory system with various scenarios."""
    print("🧪 Testing Enhanced Memory System")
    print("="*50)
    
    # Create test session
    test_session = f"memory_test_{datetime.now().strftime('%H%M%S')}"
    
    # Test 1: Basic memory operations
    print("1. Testing basic memory operations...")
    memory_manager = EnhancedMemoryManager(
        session_id=test_session,
        agent_name="test_agent",
        max_tokens=1000,  # Small limit for testing
        summarize_threshold=0.6  # 600 tokens
    )
    
    # Add messages to trigger summarization
    for i in range(10):
        long_message = f"Test message {i+1}: " + "This is a long test message that contains substantial content to simulate real agent conversations. " * 5
        memory_manager.add_message(long_message, "assistant")
        
        stats = memory_manager.get_memory_stats()
        print(f"   Message {i+1}: {stats['estimated_tokens']:,} tokens ({stats['usage_percentage']:.1f}%)")
        
        if stats['summarization_count'] > 0:
            print(f"   🔄 Summarization triggered! Count: {stats['summarization_count']}")
            break
    
    # Test 2: Force summarization
    print("\n2. Testing manual summarization...")
    memory_manager.force_summarize()
    final_stats = memory_manager.get_memory_stats()
    print(f"   Final tokens: {final_stats['estimated_tokens']:,}")
    print(f"   Summarizations: {final_stats['summarization_count']}")
    
    # Test 3: Memory retrieval
    print("\n3. Testing memory retrieval...")
    buffer = memory_manager.get_conversation_buffer()
    print(f"   Buffer length: {len(buffer)} characters")
    
    recent = memory_manager.get_recent_messages(3)
    print(f"   Recent messages: {len(recent)}")
    
    print("✅ Memory system test completed!")

def demonstrate_large_context():
    """Demonstrate the large context window capabilities."""
    print("🎯 Demonstrating Large Context Window (200k tokens)")
    print("="*60)
    
    demo_session = f"large_context_demo_{datetime.now().strftime('%H%M%S')}"
    
    # Create agent with large context
    memory_manager = EnhancedMemoryManager(
        session_id=demo_session,
        agent_name="large_context_demo",
        max_tokens=200000,
        summarize_threshold=0.7
    )
    
    # Simulate a long migration conversation
    migration_scenarios = [
        "Analyzing Spring Boot project with 50+ dependencies",
        "Identified 15 javax.* packages that need migration to jakarta.*",
        "Found deprecated JUnit 4 tests across 25 test classes",
        "Detected Spring Security configuration using old API patterns",
        "Maven Central shows newer versions available for 12 dependencies",
        "OpenRewrite recipes available for automatic javax→jakarta migration",
        "Test failures in 8 classes due to package import changes",
        "Build errors related to Spring Boot 3.x configuration changes",
        "Successfully migrated 60% of javax imports to jakarta",
        "Remaining test failures require manual intervention",
    ]
    
    print("Simulating extended migration conversation...")
    
    for i, scenario in enumerate(migration_scenarios):
        # Create detailed message for each scenario
        detailed_message = f"""
        Migration Step {i+1}: {scenario}
        
        Detailed Analysis:
        - Technical details about the migration step
        - Code examples and configuration changes needed
        - Potential issues and resolution strategies
        - Dependencies affected and version recommendations
        - Testing approach and validation criteria
        - Risk assessment and rollback procedures
        
        This message simulates the kind of detailed technical discussion
        that would occur during a real Java migration project. Each step
        involves substantial technical details, code analysis, and 
        decision-making that would generate significant conversation history.
        
        """ + "Additional technical details and context. " * 50  # Pad to increase size
        
        memory_manager.add_message(detailed_message, "assistant")
        
        stats = memory_manager.get_memory_stats()
        print(f"Step {i+1:2d}: {stats['estimated_tokens']:6,} tokens ({stats['usage_percentage']:5.1f}%) - {stats['summarization_count']} summaries")
        
        if stats['usage_percentage'] > 50:  # Show more detail as we approach limits
            print(f"        Ready for summarization: {stats['ready_for_summarization']}")
    
    final_stats = memory_manager.get_memory_stats()
    print(f"\n📊 Final Statistics:")
    print(f"   • Total tokens used: {final_stats['estimated_tokens']:,}")
    print(f"   • Memory utilization: {final_stats['usage_percentage']:.1f}%")
    print(f"   • Summarizations performed: {final_stats['summarization_count']}")
    print(f"   • Messages in memory: {final_stats['messages_count']}")
    
    if final_stats['summarization_count'] > 0:
        print(f"   🔄 Automatic summarization successfully managed large context!")
    else:
        print(f"   📈 Still within single context window - no summarization needed")

def get_session_memory_report(session_id: str) -> Dict[str, Any]:
    """Generate comprehensive memory report for a session."""
    redis_client = redis.from_url(REDIS_URL)
    agents = ["analysis", "code_migration", "validation"]
    
    report = {
        "session_id": session_id,
        "timestamp": datetime.now().isoformat(),
        "agents": {},
        "totals": {
            "total_tokens": 0,
            "total_summarizations": 0,
            "total_messages": 0
        }
    }
    
    for agent_name in agents:
        memory_key = f"memory_stats:{session_id}:{agent_name}"
        stats_data = redis_client.get(memory_key)
        
        if stats_data:
            stats = json.loads(stats_data)
            
            # Get summarization events
            event_key = f"summarization_log:{session_id}:{agent_name}"
            events = redis_client.lrange(event_key, 0, -1)
            summarization_events = [json.loads(event) for event in events]
            
            agent_report = {
                "memory_stats": stats,
                "summarization_events": summarization_events,
                "health": {
                    "within_limits": stats.get('usage_percentage', 0) < 100,
                    "efficient_usage": stats.get('usage_percentage', 0) < 80,
                    "summarization_working": len(summarization_events) > 0
                }
            }
            
            report["agents"][agent_name] = agent_report
            
            # Update totals
            report["totals"]["total_tokens"] += stats.get('estimated_tokens', 0)
            report["totals"]["total_summarizations"] += stats.get('summarization_count', 0)
            report["totals"]["total_messages"] += stats.get('messages_count', 0)
    
    return report

print("✅ Memory monitoring and testing utilities implemented!")
print("\n📚 Memory Testing Functions:")
print("1. monitor_memory_usage('session_id')  # Monitor real-time memory usage")
print("2. test_memory_system()  # Test memory management with small limits")
print("3. demonstrate_large_context()  # Demo 200k token context window")
print("4. get_session_memory_report('session_id')  # Comprehensive memory report")
print("\n🎯 Key Improvements Made:")
print("   • 200,000 token context windows (100x larger than before!)")
print("   • Proactive summarization at 60-70% usage")
print("   • Real-time token usage monitoring")
print("   • Comprehensive conversation history preservation")
print("   • Redis-backed persistence with 24h retention")
print("   • Per-agent memory isolation")
print("   • Automatic summarization logging and analytics")

✅ Memory monitoring and testing utilities implemented!

📚 Memory Testing Functions:
1. monitor_memory_usage('session_id')  # Monitor real-time memory usage
2. test_memory_system()  # Test memory management with small limits
3. demonstrate_large_context()  # Demo 200k token context window
4. get_session_memory_report('session_id')  # Comprehensive memory report

🎯 Key Improvements Made:
   • 200,000 token context windows (100x larger than before!)
   • Proactive summarization at 60-70% usage
   • Real-time token usage monitoring
   • Comprehensive conversation history preservation
   • Redis-backed persistence with 24h retention
   • Per-agent memory isolation
   • Automatic summarization logging and analytics


In [73]:
run_configurable_migration_demo('/Users/abhisheksankar/Desktop/PyTorch-Notebooks/test/xsync', False)  # Default config



🎛️ CONFIGURABLE JAVA MIGRATION SYSTEM
🆕 NEW FEATURES:
  • Configurable migration phases
  • Advanced deprecation detection with LLM analysis
  • Automatic replacement of deprecated APIs/patterns
  • Maven plugin deprecation scanning
  • Phase-specific error handling and retry logic
  • Runtime phase reconfiguration
📋 Using default configuration with all phases enabled
📊 Enhanced Memory initialized for analysis:
   • Max tokens: 200,000
   • Summarize at: 140,000 tokens (70%)

📂 Project Path: /Users/abhisheksankar/Desktop/PyTorch-Notebooks/test/xsync
🆔 Session ID: migration_20250725_202807
🔗 Redis URL: redis://localhost:6379
📋 Enabled Phases: ['analysis', 'deprecation_detection', 'dependency_update', 'code_migration', 'testing_validation', 'final_cleanup']
🔍 Deprecation Detection:
   • deep_scan: True
   • include_transitive_deps: True
   • check_plugin_compatibility: True
   • analyze_code_patterns: True
🚀 Starting migration for project: /Users/abhisheksankar/Desktop/PyTorch-Notebooks

/var/folders/rm/xfn65lb11xz98zmb7hznp8nc0000gn/T/ipykernel_34558/971866704.py:54: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  self.redis_client.set(state_key, state.json(), ex=86400)  # 24h expiry
